# Biodiversity and Native Planting Planner

## Master Dataset Development

This notebook creates a master native-plant dataset by combining:

- MoBot state-level native plant data
- Lady Bird Johnson plant attributes
- USDA plant taxonomy
- Pollinator–plant interaction records
- Propagation corrections
- Bloom-period standardization
- Height standardization

The final dataset will support native plant recommendations based on:

- U.S. state
- Sunlight
- Water availability
- Planting space
- Bloom period
- Pollinator support# Biodiversity and Native Planting Planner

## Master Dataset Development

This notebook creates a master native-plant dataset by combining:

- MoBot state-level native plant data
- Lady Bird Johnson plant attributes
- USDA plant taxonomy
- Pollinator–plant interaction records
- Propagation corrections
- Bloom-period standardization
- Height standardization

The final dataset will support native plant recommendations based on:

- U.S. state
- Sunlight
- Water availability
- Planting space
- Bloom period
- Pollinator support

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 120)

print("Pandas display settings configured.")

Pandas display settings configured.


In [3]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REVIEW_DIR = PROJECT_ROOT / "data" / "review"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Review files:", REVIEW_DIR)

Project root: C:\Users\hp\OneDrive\Desktop\biodiversity-planner
Raw data: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw
Processed data: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\processed
Review files: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\review


In [4]:
source_files = {
    "MoBot": RAW_DIR / "MoBot.xlsx",
    "Ladybird": RAW_DIR / "Ladybird.csv",
    "Pollinator interactions": RAW_DIR / "main_pollinator_plant.csv",
    "Propagation fixes": RAW_DIR / "Propagation_Fixes.csv",
    "Height standardization": RAW_DIR / "Heights_Standardization.csv",
    "Bloom standardization": RAW_DIR / "BloomPeriod_Standardization.csv",
    "USDA plant list": RAW_DIR / "plantlst.txt",
}

file_check = pd.DataFrame(
    {
        "dataset": source_files.keys(),
        "path": [str(path) for path in source_files.values()],
        "exists": [path.exists() for path in source_files.values()],
    }
)

file_check

,dataset,path,exists
0,MoBot,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\MoBot.xlsx,True
1,Ladybird,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\Ladybird.csv,True
2,Pollinator interactions,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\main_pollinator_plant.csv,True
3,Propagation fixes,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\Propagation_Fixes.csv,True
4,Height standardization,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\Heights_Standardization.csv,True
5,Bloom standardization,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\BloomPeriod_Standardization.csv,True
6,USDA plant list,C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\raw\plantlst.txt,True


In [5]:
mobot_path = source_files["MoBot"]

mobot_workbook = pd.ExcelFile(mobot_path)

print("Number of worksheets:", len(mobot_workbook.sheet_names))
print("\nWorksheet names:")
print(mobot_workbook.sheet_names)

Number of worksheets: 50

Worksheet names:
['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']


In [6]:
first_sheet = mobot_workbook.sheet_names[0]

mobot_sample = pd.read_excel(
    mobot_path,
    sheet_name=first_sheet,
)

print("Worksheet:", first_sheet)
print("Shape:", mobot_sample.shape)

mobot_sample.head()

Worksheet: Alabama
Shape: (645, 10)


,Taxon,Common name,Plant Type,Sun,Moisture,Maintenance,Zone From,Zone To,Bloom From,Bloom To
0,Acer negundo,boxelder,Tree,Full sun,Medium to wet,Low,2.0,10.0,March,April
1,Acer rubrum,red maple,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,March,April
2,Acer saccharinum,silver maple,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,March,March
3,Acer saccharum,sugar maple,Tree,Full sun to part shade,Medium,Medium,3.0,8.0,April,April
4,Acer saccharum subsp. nigrum,black maple,Tree,Full sun to part shade,Medium,Medium,4.0,8.0,April,April


In [7]:
mobot_sample.columns.tolist()

['Taxon',
 'Common name',
 'Plant Type',
 'Sun',
 'Moisture',
 'Maintenance',
 'Zone From',
 'Zone To',
 'Bloom From',
 'Bloom To']

In [8]:
mobot_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Taxon        645 non-null    object 
 1   Common name  645 non-null    object 
 2   Plant Type   645 non-null    object 
 3   Sun          645 non-null    object 
 4   Moisture     645 non-null    object 
 5   Maintenance  645 non-null    object 
 6   Zone From    644 non-null    float64
 7   Zone To      644 non-null    float64
 8   Bloom From   645 non-null    object 
 9   Bloom To     645 non-null    object 
dtypes: float64(2), object(8)
memory usage: 50.5+ KB


In [9]:
mobot_sample.isna().sum().sort_values(ascending=False)

Zone From      1
Zone To        1
Taxon          0
Common name    0
Plant Type     0
Sun            0
Moisture       0
Maintenance    0
Bloom From     0
Bloom To       0
dtype: int64

In [10]:
mobot_frames = []

for state in mobot_workbook.sheet_names:
    state_df = pd.read_excel(
        mobot_path,
        sheet_name=state,
    )

    state_df["Native State"] = state
    mobot_frames.append(state_df)

mobot_raw = pd.concat(
    mobot_frames,
    ignore_index=True,
)

print("Combined MoBot shape:", mobot_raw.shape)
print("Number of states:", mobot_raw["Native State"].nunique())

mobot_raw.head()

Combined MoBot shape: (20091, 11)
Number of states: 50


,Taxon,Common name,Plant Type,Sun,Moisture,Maintenance,Zone From,Zone To,Bloom From,Bloom To,Native State
0,Acer negundo,boxelder,Tree,Full sun,Medium to wet,Low,2.0,10.0,March,April,Alabama
1,Acer rubrum,red maple,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,March,April,Alabama
2,Acer saccharinum,silver maple,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,March,March,Alabama
3,Acer saccharum,sugar maple,Tree,Full sun to part shade,Medium,Medium,3.0,8.0,April,April,Alabama
4,Acer saccharum subsp. nigrum,black maple,Tree,Full sun to part shade,Medium,Medium,4.0,8.0,April,April,Alabama


In [11]:
mobot_raw["Native State"].value_counts().sort_index()

Native State
Alabama           645
Alaska             68
Arizona           136
Arkansas          622
California        141
Colorado          184
Connecticut       451
Delaware          435
Florida           439
Georgia           650
Hawaii             17
Idaho             113
Illinois          641
Indiana           596
Iowa              472
Kansas            448
Kentucky          606
Louisiana         490
Maine             358
Maryland          549
Massachusetts     391
Michigan          518
Minnesota         418
Mississippi       534
Missouri          643
Montana           157
Nebraska          339
Nevada             80
New Hampshire     366
New Jersey        496
New Mexico        199
New York          520
North Carolina    618
North Dakota      219
Ohio              589
Oklahoma          531
Oregon            124
Pennsylvania      559
Rhode Island      367
South Carolina    563
South Dakota      269
Tennessee         650
Texas             497
Utah              114
Vermont           3

In [12]:
mobot_raw.sample(
    n=min(10, len(mobot_raw)),
    random_state=42,
)

,Taxon,Common name,Plant Type,Sun,Moisture,Maintenance,Zone From,Zone To,Bloom From,Bloom To,Native State
8900,Stylophorum diphyllum,celandine poppy,Herbaceous perennial,Part shade to full shade,Medium to wet,High,4.0,9.0,April,June,Michigan
2812,Elodea canadensis,Canadian pondweed,Herbaceous perennial,Full sun,Wet,Medium,4.0,10.0,July,September,Florida
1606,Vaccinium macrocarpon,American cranberry,Broadleaf evergreen,Full sun,Medium to wet,Medium,3.0,7.0,May,June,California
10599,Berberis aquifolium,holly-leaved barberry,Broadleaf evergreen,Part shade to full shade,Medium,Medium,5.0,8.0,April,April,Montana
19846,Silphium laciniatum,compass plant,Herbaceous perennial,Full sun,Medium,Low,3.0,8.0,July,September,Wisconsin
1114,Helianthus annuus,common sunflower,Annual,Full sun,Dry to medium,Low,2.0,11.0,July,August,Arkansas
7912,Rhododendron atlanticum,deciduous azalea,Deciduous shrub,Part shade,Medium,Low,6.0,8.0,April,April,Maryland
1712,Juniperus communis,common juniper,Needled evergreen,Full sun,Medium,Low,2.0,7.0,Non-flowering,Non-flowering,Colorado
12053,Campsis radicans f. flava,trumpetcreeper,Vine,Full sun to part shade,Medium,High,4.0,9.0,July,August,New Mexico
13902,Nyssa sylvatica,black gum,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,May,June,Ohio


In [13]:
ladybird_raw = pd.read_csv(
    source_files["Ladybird"],
    encoding="cp1252",
    encoding_errors="replace",
    low_memory=False,
)

print(ladybird_raw.shape)
ladybird_raw.head()

(4049, 20)


,SpeciesName,CommonName,USDAcode,USDAstatus,Duration,Habit,SizeNotes,BloomColor,BloomTime,Distribution,WaterUse,LightRequirements,SoilMoisture,SolpH,SoilDescription,UseWildlife,InterestingFoliage,FragrantFoliage,CommercialAvail,PropagationDescription
0,Abies amabilis,"Pacific Silver Fir, Cascade Fir, Lovely Fir, White Fir, Red fir",ABAM,"USDA Native Status: L48 (N), AK (N), CAN (N)",NaN,Tree,Up to about 200 feet tall.,Yellow,"Apr , May , Jun",Pacific Coast from extreme SE. Alaska south to W. Oregon; local in NW. California; to 1000' (305 m) in north; to 6000' (1829 m) in south.,Medium,Shade,Moist,Acidic (pH<6.8),Loam,NaN,yes,yes,NaN,NaN
1,Abies balsamea,"Balsam Fir, Blister Pine, Northern Balsam",ABBA,"USDA Native Status: L48 (N), CAN (N), SPM (N)",Perennial,Tree,Up to about 75 feet tall.,"Yellow , Green , Purple , Brown","Sep , Oct , Nov","Lab. & Nf. to MN & s. Man., s. to VA & n.e. IA; in North on low, swampy ground to well-drained uplands; in South above 3600 ft.",Medium,"Sun , Part Shade , Shade",Moist,Acidic (pH<6.8),"Well-drained, acid, moist soils.",Songbirds and squirrels eat seed and deer browse foliage. Deer and moose browse the foliage in winter.,yes,yes,yes,"Abies spp. are best propagated by means of seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."
2,Abies concolor,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",ABCO,USDA Native Status: L48 (N),Perennial,Tree,"Up to about 130 feet tall, spread up to about 60 feet.",Red,"Apr , May , Jun","S.w. ID & s.e. OR, w. to mts. of c. CO, & s. to s. CA, AZ & NM",Medium,"Sun , Part Shade",NaN,NaN,"Well-drained, gravelly or sandy-loam soils.","The winged seeds of this and other firs are eaten by songbirds and various mammals, especially squirrels and chipmunks. Deer and grouse feed on th...",yes,NaN,yes,"Seed is the easiest method of propagation. In nature, Abies seeds often germinate on melting snow fields. Cuttings should be taken from December t..."
3,Abies fraseri,"Fraser Fir, She-balsam",ABFR,USDA Native Status: L48 (N),Perennial,Tree,Up to about 75 feet tall.,Purple,Apr,"Appalachian Mountains in sw. Virginia, w. North Carolina, and e. Tennessee; at 4000-6000'",Medium,Shade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Abies grandis,"Grand Fir, Giant Fir",ABGR,"USDA Native Status: L48 (N), CAN (N)",NaN,Tree,Up to more than 200 feet tall.,"White , Green","Apr , May","S. B.C. to w. MT, s. to n.w. CA",Medium,"Part Shade , Shade","Dry , Moist",NaN,Well-drained soils.,NaN,NaN,yes,yes,"Abies spp. are best propagated by seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."


In [14]:
ladybird_raw.columns.tolist()

['SpeciesName',
 'CommonName',
 'USDAcode',
 'USDAstatus',
 'Duration',
 'Habit',
 'SizeNotes',
 'BloomColor',
 'BloomTime',
 'Distribution',
 'WaterUse',
 'LightRequirements',
 'SoilMoisture',
 'SolpH',
 'SoilDescription',
 'UseWildlife',
 'InterestingFoliage',
 'FragrantFoliage',
 'CommercialAvail',
 'PropagationDescription']

In [15]:
ladybird_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4049 entries, 0 to 4048
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   SpeciesName             4049 non-null   object
 1   CommonName              4049 non-null   object
 2   USDAcode                4049 non-null   object
 3   USDAstatus              4049 non-null   object
 4   Duration                3010 non-null   object
 5   Habit                   4049 non-null   object
 6   SizeNotes               3782 non-null   object
 7   BloomColor              4019 non-null   object
 8   BloomTime               3881 non-null   object
 9   Distribution            1949 non-null   object
 10  WaterUse                1366 non-null   object
 11  LightRequirements       2255 non-null   object
 12  SoilMoisture            1904 non-null   object
 13  SolpH                   517 non-null    object
 14  SoilDescription         1677 non-null   object
 15  UseW

In [16]:
ladybird_raw.head()

,SpeciesName,CommonName,USDAcode,USDAstatus,Duration,Habit,SizeNotes,BloomColor,BloomTime,Distribution,WaterUse,LightRequirements,SoilMoisture,SolpH,SoilDescription,UseWildlife,InterestingFoliage,FragrantFoliage,CommercialAvail,PropagationDescription
0,Abies amabilis,"Pacific Silver Fir, Cascade Fir, Lovely Fir, White Fir, Red fir",ABAM,"USDA Native Status: L48 (N), AK (N), CAN (N)",NaN,Tree,Up to about 200 feet tall.,Yellow,"Apr , May , Jun",Pacific Coast from extreme SE. Alaska south to W. Oregon; local in NW. California; to 1000' (305 m) in north; to 6000' (1829 m) in south.,Medium,Shade,Moist,Acidic (pH<6.8),Loam,NaN,yes,yes,NaN,NaN
1,Abies balsamea,"Balsam Fir, Blister Pine, Northern Balsam",ABBA,"USDA Native Status: L48 (N), CAN (N), SPM (N)",Perennial,Tree,Up to about 75 feet tall.,"Yellow , Green , Purple , Brown","Sep , Oct , Nov","Lab. & Nf. to MN & s. Man., s. to VA & n.e. IA; in North on low, swampy ground to well-drained uplands; in South above 3600 ft.",Medium,"Sun , Part Shade , Shade",Moist,Acidic (pH<6.8),"Well-drained, acid, moist soils.",Songbirds and squirrels eat seed and deer browse foliage. Deer and moose browse the foliage in winter.,yes,yes,yes,"Abies spp. are best propagated by means of seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."
2,Abies concolor,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",ABCO,USDA Native Status: L48 (N),Perennial,Tree,"Up to about 130 feet tall, spread up to about 60 feet.",Red,"Apr , May , Jun","S.w. ID & s.e. OR, w. to mts. of c. CO, & s. to s. CA, AZ & NM",Medium,"Sun , Part Shade",NaN,NaN,"Well-drained, gravelly or sandy-loam soils.","The winged seeds of this and other firs are eaten by songbirds and various mammals, especially squirrels and chipmunks. Deer and grouse feed on th...",yes,NaN,yes,"Seed is the easiest method of propagation. In nature, Abies seeds often germinate on melting snow fields. Cuttings should be taken from December t..."
3,Abies fraseri,"Fraser Fir, She-balsam",ABFR,USDA Native Status: L48 (N),Perennial,Tree,Up to about 75 feet tall.,Purple,Apr,"Appalachian Mountains in sw. Virginia, w. North Carolina, and e. Tennessee; at 4000-6000'",Medium,Shade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Abies grandis,"Grand Fir, Giant Fir",ABGR,"USDA Native Status: L48 (N), CAN (N)",NaN,Tree,Up to more than 200 feet tall.,"White , Green","Apr , May","S. B.C. to w. MT, s. to n.w. CA",Medium,"Part Shade , Shade","Dry , Moist",NaN,Well-drained soils.,NaN,NaN,yes,yes,"Abies spp. are best propagated by seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."


In [17]:
ladybird_missing = (
    ladybird_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

ladybird_missing["missing_percent"] = (
    ladybird_missing["missing_count"]
    / len(ladybird_raw)
    * 100
).round(2)

ladybird_missing

,missing_count,missing_percent
FragrantFoliage,3969,98.02
InterestingFoliage,3732,92.17
SolpH,3532,87.23
UseWildlife,3369,83.21
PropagationDescription,2932,72.41
CommercialAvail,2794,69.00
WaterUse,2683,66.26
SoilDescription,2372,58.58
SoilMoisture,2145,52.98
Distribution,2100,51.86


In [18]:
ladybird_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4049 entries, 0 to 4048
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   SpeciesName             4049 non-null   object
 1   CommonName              4049 non-null   object
 2   USDAcode                4049 non-null   object
 3   USDAstatus              4049 non-null   object
 4   Duration                3010 non-null   object
 5   Habit                   4049 non-null   object
 6   SizeNotes               3782 non-null   object
 7   BloomColor              4019 non-null   object
 8   BloomTime               3881 non-null   object
 9   Distribution            1949 non-null   object
 10  WaterUse                1366 non-null   object
 11  LightRequirements       2255 non-null   object
 12  SoilMoisture            1904 non-null   object
 13  SolpH                   517 non-null    object
 14  SoilDescription         1677 non-null   object
 15  UseW

In [19]:
pollinator_path = source_files["Pollinator interactions"]

pollinator_raw = pd.read_csv(
    pollinator_path,
    low_memory=False,
)

print("Pollinator dataset shape:", pollinator_raw.shape)

pollinator_raw.head()

Pollinator dataset shape: (67954, 23)


,catalog_number,pollinator_family,pollinator_genus,pollinator_species,pollinator_sex,plant_sp_code,plant_species,collection_method,pantrap_colour,collector_number,day_collected,month_collected,year_collected,time_of_day_collected,location_description,location_name,habitat,latitude,longitude,basis_of_record,location_specimen,study_director,study
0,SFU705915,Tachinidae,Tachinidae,NaN,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
1,SFU705916,Halictidae,Lasioglossum,sp. 1,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
2,SFU705917,Andrenidae,Andrena,nigrocaerulea,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
3,SFU705918,Sarcophagidae,Sarcophagidae,NaN,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
4,SFU705919,Halictidae,Lasioglossum,incompletum,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007


In [20]:
pollinator_raw.columns.tolist()

['catalog_number',
 'pollinator_family',
 'pollinator_genus',
 'pollinator_species',
 'pollinator_sex',
 'plant_sp_code',
 'plant_species',
 'collection_method',
 'pantrap_colour',
 'collector_number',
 'day_collected',
 'month_collected',
 'year_collected',
 'time_of_day_collected',
 'location_description',
 'location_name',
 'habitat',
 'latitude',
 'longitude',
 'basis_of_record',
 'location_specimen',
 'study_director',
 'study']

In [21]:
pollinator_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67954 entries, 0 to 67953
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   catalog_number         67954 non-null  object 
 1   pollinator_family      67946 non-null  object 
 2   pollinator_genus       67954 non-null  object 
 3   pollinator_species     61050 non-null  object 
 4   pollinator_sex         55655 non-null  object 
 5   plant_sp_code          34949 non-null  object 
 6   plant_species          34949 non-null  object 
 7   collection_method      67954 non-null  object 
 8   pantrap_colour         26419 non-null  object 
 9   collector_number       67954 non-null  int64  
 10  day_collected          67954 non-null  int64  
 11  month_collected        67954 non-null  object 
 12  year_collected         67954 non-null  int64  
 13  time_of_day_collected  5322 non-null   object 
 14  location_description   67954 non-null  object 
 15  lo

In [22]:
plant_column = "plant_species"

if plant_column in pollinator_raw.columns:
    print(
        "Records with plant names:",
        pollinator_raw[plant_column].notna().sum(),
    )

    print(
        "Unique plant names:",
        pollinator_raw[plant_column].nunique(),
    )
else:
    print(
        f"Column '{plant_column}' was not found."
    )

Records with plant names: 34949
Unique plant names: 472


In [23]:
propagation_fixes_raw = pd.read_csv(
    source_files["Propagation fixes"],
    low_memory=False,
)

height_standardization_raw = pd.read_csv(
    source_files["Height standardization"],
    low_memory=False,
)

bloom_standardization_raw = pd.read_csv(
    source_files["Bloom standardization"],
    low_memory=False,
)

print(
    "Propagation fixes:",
    propagation_fixes_raw.shape,
)

print(
    "Height standardization:",
    height_standardization_raw.shape,
)

print(
    "Bloom standardization:",
    bloom_standardization_raw.shape,
)

Propagation fixes: (1087, 4)
Height standardization: (361, 2)
Bloom standardization: (118, 2)


In [24]:
display(propagation_fixes_raw.head())
display(height_standardization_raw.head())
display(bloom_standardization_raw.head())

,AcceptedName,USDAcode,PropagationDescription.L,Propogation Description 2
0,Aristolochia macrophylla,ARMA7,"Increase by layering, division, July cuttings, or by seed sown outdoor in fall.","Increase by layering, division, July cuttings, or by seed sown outdoor in fall."
1,Chionanthus virginicus,CHVI3,NaN,NaN
2,Abies balsamea,ABBA,"Abies spp. are best propagated by means of seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields.","Abies spp. are best propagated by means of seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."
3,Celastrus scandens,CESC,"Sow seeds in fall or stratify and sow in spring. Bittersweet can also be propagated by root cuttings, layers, suckers, hardwood and softwood cutti...","Sow seeds in fall or stratify and sow in spring. Bittersweet can also be propagated by root cuttings, layers, suckers, hardwood and softwood cutti..."
4,Clethra alnifolia,CLAL3,"Propagate by seed or softwood cuttings, with or without hormone treatment, under mist. Sow seed on sand.","Propagate by seed or softwood cuttings, with or without hormone treatment, under mist. Sow seed on sand."


,Height..ft.,fix
0,15–30,15–30
1,12–35,12–35
2,60,60
3,48,48
4,4–9,4–9


,original,fix
0,May to June,May-June
1,Non-flowering,Non-flowering
2,July to August,July-August
3,Apr–May,April-May
4,"May , Jun , Jul",May-July


In [25]:
print(
    "Propagation columns:",
    propagation_fixes_raw.columns.tolist(),
)

print(
    "\nHeight columns:",
    height_standardization_raw.columns.tolist(),
)

print(
    "\nBloom columns:",
    bloom_standardization_raw.columns.tolist(),
)

Propagation columns: ['AcceptedName', 'USDAcode', 'PropagationDescription.L', 'Propogation Description 2']

Height columns: ['Height..ft.', 'fix']

Bloom columns: ['original', 'fix']


In [26]:
usda_path = source_files["USDA plant list"]

usda_raw = pd.read_csv(
    usda_path,
    encoding="utf-8-sig",
    low_memory=False,
)

print("USDA plant list shape:", usda_raw.shape)

usda_raw.head()

USDA plant list shape: (93157, 5)


,Symbol,Synonym Symbol,Scientific Name with Author,Common Name,Family
0,ABAB,NaN,Abutilon abutiloides (Jacq.) Garcke ex Hochr.,shrubby Indian mallow,Malvaceae
1,ABAB,ABAM5,Abutilon americanum (L.) Sweet,NaN,NaN
2,ABAB,ABJA,Abutilon jacquinii G. Don,NaN,NaN
3,ABAB,ABLI,Abutilon lignosum (Cav.) G. Don,NaN,NaN
4,ABAB70,NaN,Abietinella abietina (Hedw.) Fleisch.,abietinella moss,Thuidiaceae


In [27]:
usda_raw.columns.tolist()

['Symbol',
 'Synonym Symbol',
 'Scientific Name with Author',
 'Common Name',
 'Family']

In [28]:
usda_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93157 entries, 0 to 93156
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Symbol                       93157 non-null  object
 1   Synonym Symbol               44163 non-null  object
 2   Scientific Name with Author  93157 non-null  object
 3   Common Name                  43776 non-null  object
 4   Family                       48994 non-null  object
dtypes: object(5)
memory usage: 3.6+ MB


In [29]:
dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "MoBot",
            "rows": len(mobot_raw),
            "columns": mobot_raw.shape[1],
        },
        {
            "dataset": "Ladybird",
            "rows": len(ladybird_raw),
            "columns": ladybird_raw.shape[1],
        },
        {
            "dataset": "Pollinator interactions",
            "rows": len(pollinator_raw),
            "columns": pollinator_raw.shape[1],
        },
        {
            "dataset": "Propagation fixes",
            "rows": len(propagation_fixes_raw),
            "columns": propagation_fixes_raw.shape[1],
        },
        {
            "dataset": "Height standardization",
            "rows": len(height_standardization_raw),
            "columns": height_standardization_raw.shape[1],
        },
        {
            "dataset": "Bloom standardization",
            "rows": len(bloom_standardization_raw),
            "columns": bloom_standardization_raw.shape[1],
        },
        {
            "dataset": "USDA plant list",
            "rows": len(usda_raw),
            "columns": usda_raw.shape[1],
        },
    ]
)

dataset_summary

,dataset,rows,columns
0,MoBot,20091,11
1,Ladybird,4049,20
2,Pollinator interactions,67954,23
3,Propagation fixes,1087,4
4,Height standardization,361,2
5,Bloom standardization,118,2
6,USDA plant list,93157,5


In [30]:
mobot_raw.to_pickle(
    PROCESSED_DIR / "checkpoint_mobot_raw.pkl"
)

ladybird_raw.to_pickle(
    PROCESSED_DIR / "checkpoint_ladybird_raw.pkl"
)

pollinator_raw.to_pickle(
    PROCESSED_DIR / "checkpoint_pollinator_raw.pkl"
)

print("Raw-load checkpoints saved.")

Raw-load checkpoints saved.


# Phase 2

## Data Cleaning and Standardization

Before merging datasets, we standardize text, column names, and missing values to ensure reliable joins.

In [31]:
import numpy as np

In [32]:
MISSING_VALUES = {
    "",
    " ",
    "NA",
    "N/A",
    "NULL",
    "null",
    "None",
    "none",
    "Unknown",
    "unknown",
    "-",
}

In [33]:
def clean_text(value):
    """
    Standardize text values.
    """

    if pd.isna(value):
        return pd.NA

    value = unicodedata.normalize("NFKC", str(value))
    value = value.strip()

    if value in MISSING_VALUES:
        return pd.NA

    return value

In [34]:
def normalize_scientific_name(name):

    name = clean_text(name)

    if pd.isna(name):
        return pd.NA

    return name.lower()

In [35]:
def normalize_usda_code(code):

    code = clean_text(code)

    if pd.isna(code):
        return pd.NA

    return code.upper()

In [36]:
samples = [
    " Quercus alba ",
    "QUERCUS ALBA",
    "unknown",
    "",
    np.nan,
]

for item in samples:
    print(item, " --> ", clean_text(item))

normalize_scientific_name(" Quercus Alba ")
normalize_usda_code("abfr")

 Quercus alba   -->  Quercus alba
QUERCUS ALBA  -->  QUERCUS ALBA
unknown  -->  <NA>
  -->  <NA>
nan  -->  <NA>


'ABFR'

In [37]:
normalize_scientific_name(" Quercus Alba ")

'quercus alba'

In [38]:
mobot = mobot_raw.copy()

print(mobot.shape)
mobot.head()

(20091, 11)


,Taxon,Common name,Plant Type,Sun,Moisture,Maintenance,Zone From,Zone To,Bloom From,Bloom To,Native State
0,Acer negundo,boxelder,Tree,Full sun,Medium to wet,Low,2.0,10.0,March,April,Alabama
1,Acer rubrum,red maple,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,March,April,Alabama
2,Acer saccharinum,silver maple,Tree,Full sun to part shade,Medium to wet,Low,3.0,9.0,March,March,Alabama
3,Acer saccharum,sugar maple,Tree,Full sun to part shade,Medium,Medium,3.0,8.0,April,April,Alabama
4,Acer saccharum subsp. nigrum,black maple,Tree,Full sun to part shade,Medium,Medium,4.0,8.0,April,April,Alabama


In [39]:
mobot = mobot.rename(
    columns={
        "Taxon": "scientific_name",
        "Common name": "common_name",
        "Plant Type": "plant_type",
        "Sun": "sunlight",
        "Moisture": "moisture",
        "Maintenance": "maintenance",
        "Zone From": "zone_min",
        "Zone To": "zone_max",
        "Bloom From": "bloom_start",
        "Bloom To": "bloom_end",
        "Native State": "native_state",
    }
)

mobot.columns.tolist()

['scientific_name',
 'common_name',
 'plant_type',
 'sunlight',
 'moisture',
 'maintenance',
 'zone_min',
 'zone_max',
 'bloom_start',
 'bloom_end',
 'native_state']

In [40]:
mobot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20091 entries, 0 to 20090
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   scientific_name  20091 non-null  object 
 1   common_name      20067 non-null  object 
 2   plant_type       20091 non-null  object 
 3   sunlight         20091 non-null  object 
 4   moisture         20091 non-null  object 
 5   maintenance      20091 non-null  object 
 6   zone_min         20058 non-null  float64
 7   zone_max         20058 non-null  float64
 8   bloom_start      20091 non-null  object 
 9   bloom_end        20091 non-null  object 
 10  native_state     20091 non-null  object 
dtypes: float64(2), object(9)
memory usage: 1.7+ MB


In [41]:
text_columns = mobot.select_dtypes(include="object").columns

text_columns

Index(['scientific_name', 'common_name', 'plant_type', 'sunlight', 'moisture', 'maintenance', 'bloom_start',
       'bloom_end', 'native_state'],
      dtype='object')

In [42]:
for column in text_columns:
    mobot[column] = mobot[column].apply(clean_text)

In [43]:
mobot["scientific_name_key"] = (
    mobot["scientific_name"]
    .apply(normalize_scientific_name)
)

mobot[
    [
        "scientific_name",
        "scientific_name_key"
    ]
].head(10)

,scientific_name,scientific_name_key
0,Acer negundo,acer negundo
1,Acer rubrum,acer rubrum
2,Acer saccharinum,acer saccharinum
3,Acer saccharum,acer saccharum
4,Acer saccharum subsp. nigrum,acer saccharum subsp. nigrum
5,Achillea millefolium,achillea millefolium
6,Actaea pachypoda,actaea pachypoda
7,Actaea racemosa,actaea racemosa
8,Adiantum capillus-veneris,adiantum capillus-veneris
9,Adiantum pedatum,adiantum pedatum


In [44]:
mobot.isna().sum().sort_values(ascending=False)

zone_min               33
zone_max               33
common_name            24
scientific_name         0
plant_type              0
sunlight                0
moisture                0
maintenance             0
bloom_start             0
bloom_end               0
native_state            0
scientific_name_key     0
dtype: int64

In [45]:
mobot.duplicated().sum()

0

In [46]:
duplicates = mobot[
    mobot.duplicated(
        subset=[
            "scientific_name_key",
            "native_state"
        ],
        keep=False
    )
]

duplicates

,scientific_name,common_name,plant_type,sunlight,moisture,maintenance,zone_min,zone_max,bloom_start,bloom_end,native_state,scientific_name_key


In [47]:
print("Total rows:", len(mobot))

print(
    "Unique plants:",
    mobot["scientific_name_key"].nunique()
)

print(
    "States:",
    mobot["native_state"].nunique()
)

Total rows: 20091
Unique plants: 997
States: 50


In [48]:
mobot.to_pickle(
    PROCESSED_DIR / "checkpoint_mobot_clean.pkl"
)

print("Cleaned MoBot checkpoint saved.")

Cleaned MoBot checkpoint saved.


In [49]:
mobot["scientific_name"].sample(20, random_state=42)

8900          Stylophorum diphyllum
2812              Elodea canadensis
1606          Vaccinium macrocarpon
10599           Berberis aquifolium
19846           Silphium laciniatum
1114              Helianthus annuus
7912        Rhododendron atlanticum
1712             Juniperus communis
12053     Campsis radicans f. flava
13902               Nyssa sylvatica
11833                 Pinus strobus
7899             Quercus macrocarpa
13416           Crataegus douglasii
4590          Arnoglossum reniforme
15516                Galium boreale
18630        Rhododendron austrinum
2571                  Rosa carolina
8279            Mertensia virginica
3208            Bignonia capreolata
14826    Alnus incana subsp. rugosa
Name: scientific_name, dtype: object

In [50]:
mobot["scientific_name"].nunique()

997

In [51]:
mobot[
    [
        "scientific_name",
        "common_name",
        "native_state"
    ]
].sample(20, random_state=42)

,scientific_name,common_name,native_state
8900,Stylophorum diphyllum,celandine poppy,Michigan
2812,Elodea canadensis,Canadian pondweed,Florida
1606,Vaccinium macrocarpon,American cranberry,California
10599,Berberis aquifolium,holly-leaved barberry,Montana
19846,Silphium laciniatum,compass plant,Wisconsin
1114,Helianthus annuus,common sunflower,Arkansas
7912,Rhododendron atlanticum,deciduous azalea,Maryland
1712,Juniperus communis,common juniper,Colorado
12053,Campsis radicans f. flava,trumpetcreeper,New Mexico
13902,Nyssa sylvatica,black gum,Ohio


In [52]:
plant_distribution = (
    mobot.groupby("scientific_name_key")
    .agg(
        states=("native_state", "nunique")
    )
    .sort_values("states", ascending=False)
)

plant_distribution.head(20)

,states
scientific_name_key,
schoenoplectus tabernaemontani,50
typha latifolia,50
achillea millefolium,49
artemisia ludoviciana,49
portulaca oleracea,49
lemna minor,49
ageratina altissima,49
athyrium filix-femina,48
rhus glabra,48


In [53]:
plant_distribution.describe()

,states
count,997.000000
mean,20.151454
std,12.543024
min,1.000000
25%,9.000000
50%,20.000000
75%,30.000000
max,50.000000


In [54]:
unique_plants = (
    mobot[["scientific_name_key", "scientific_name"]]
    .drop_duplicates()
    .sort_values("scientific_name")
    .reset_index(drop=True)
)

unique_plants["plant_id"] = [
    f"PLANT-{i:05d}"
    for i in range(1, len(unique_plants) + 1)
]

unique_plants.head()

,scientific_name_key,scientific_name,plant_id
0,abies balsamea,Abies balsamea,PLANT-00001
1,abies concolor,Abies concolor,PLANT-00002
2,abies fraseri,Abies fraseri,PLANT-00003
3,abies grandis,Abies grandis,PLANT-00004
4,acacia koa,Acacia koa,PLANT-00005


In [55]:
mobot = mobot.merge(
    unique_plants,
    on=["scientific_name_key", "scientific_name"],
    how="left"
)

In [56]:
mobot[
    [
        "plant_id",
        "scientific_name",
        "native_state"
    ]
].head()

,plant_id,scientific_name,native_state
0,PLANT-00008,Acer negundo,Alabama
1,PLANT-00010,Acer rubrum,Alabama
2,PLANT-00011,Acer saccharinum,Alabama
3,PLANT-00012,Acer saccharum,Alabama
4,PLANT-00014,Acer saccharum subsp. nigrum,Alabama


In [57]:
mobot.to_pickle(
    PROCESSED_DIR / "checkpoint_mobot_clean.pkl"
)

print("Cleaned MoBot checkpoint saved.")

Cleaned MoBot checkpoint saved.


In [58]:
ladybird = ladybird_raw.copy()

print(ladybird.shape)

ladybird.head()

(4049, 20)


,SpeciesName,CommonName,USDAcode,USDAstatus,Duration,Habit,SizeNotes,BloomColor,BloomTime,Distribution,WaterUse,LightRequirements,SoilMoisture,SolpH,SoilDescription,UseWildlife,InterestingFoliage,FragrantFoliage,CommercialAvail,PropagationDescription
0,Abies amabilis,"Pacific Silver Fir, Cascade Fir, Lovely Fir, White Fir, Red fir",ABAM,"USDA Native Status: L48 (N), AK (N), CAN (N)",NaN,Tree,Up to about 200 feet tall.,Yellow,"Apr , May , Jun",Pacific Coast from extreme SE. Alaska south to W. Oregon; local in NW. California; to 1000' (305 m) in north; to 6000' (1829 m) in south.,Medium,Shade,Moist,Acidic (pH<6.8),Loam,NaN,yes,yes,NaN,NaN
1,Abies balsamea,"Balsam Fir, Blister Pine, Northern Balsam",ABBA,"USDA Native Status: L48 (N), CAN (N), SPM (N)",Perennial,Tree,Up to about 75 feet tall.,"Yellow , Green , Purple , Brown","Sep , Oct , Nov","Lab. & Nf. to MN & s. Man., s. to VA & n.e. IA; in North on low, swampy ground to well-drained uplands; in South above 3600 ft.",Medium,"Sun , Part Shade , Shade",Moist,Acidic (pH<6.8),"Well-drained, acid, moist soils.",Songbirds and squirrels eat seed and deer browse foliage. Deer and moose browse the foliage in winter.,yes,yes,yes,"Abies spp. are best propagated by means of seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."
2,Abies concolor,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",ABCO,USDA Native Status: L48 (N),Perennial,Tree,"Up to about 130 feet tall, spread up to about 60 feet.",Red,"Apr , May , Jun","S.w. ID & s.e. OR, w. to mts. of c. CO, & s. to s. CA, AZ & NM",Medium,"Sun , Part Shade",NaN,NaN,"Well-drained, gravelly or sandy-loam soils.","The winged seeds of this and other firs are eaten by songbirds and various mammals, especially squirrels and chipmunks. Deer and grouse feed on th...",yes,NaN,yes,"Seed is the easiest method of propagation. In nature, Abies seeds often germinate on melting snow fields. Cuttings should be taken from December t..."
3,Abies fraseri,"Fraser Fir, She-balsam",ABFR,USDA Native Status: L48 (N),Perennial,Tree,Up to about 75 feet tall.,Purple,Apr,"Appalachian Mountains in sw. Virginia, w. North Carolina, and e. Tennessee; at 4000-6000'",Medium,Shade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Abies grandis,"Grand Fir, Giant Fir",ABGR,"USDA Native Status: L48 (N), CAN (N)",NaN,Tree,Up to more than 200 feet tall.,"White , Green","Apr , May","S. B.C. to w. MT, s. to n.w. CA",Medium,"Part Shade , Shade","Dry , Moist",NaN,Well-drained soils.,NaN,NaN,yes,yes,"Abies spp. are best propagated by seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields."


In [59]:
pd.DataFrame({
    "Column": ladybird.columns,
    "Data Type": ladybird.dtypes.values,
    "Missing Values": ladybird.isna().sum().values
})

,Column,Data Type,Missing Values
0,SpeciesName,object,0
1,CommonName,object,0
2,USDAcode,object,0
3,USDAstatus,object,0
4,Duration,object,1039
5,Habit,object,0
6,SizeNotes,object,267
7,BloomColor,object,30
8,BloomTime,object,168
9,Distribution,object,2100


In [60]:
ladybird = ladybird.rename(columns={
    "SpeciesName": "scientific_name",
    "CommonName": "common_name",
    "USDAcode": "usda_code",
    "USDAstatus": "usda_status",
    "Duration": "duration",
    "Habit": "growth_habit",
    "SizeNotes": "size_notes",
    "BloomColor": "bloom_color",
    "BloomTime": "bloom_time",
    "Distribution": "distribution",
    "WaterUse": "water_use",
    "LightRequirements": "light_requirements",
    "SoilMoisture": "soil_moisture",
    "SolpH": "soil_ph",
    "SoilDescription": "soil_description",
    "UseWildlife": "wildlife_use",
    "InterestingFoliage": "interesting_foliage",
    "FragrantFoliage": "fragrant_foliage",
    "CommercialAvail": "commercial_availability",
    "PropagationDescription": "propagation_description"
})

In [61]:
ladybird.columns.tolist()

['scientific_name',
 'common_name',
 'usda_code',
 'usda_status',
 'duration',
 'growth_habit',
 'size_notes',
 'bloom_color',
 'bloom_time',
 'distribution',
 'water_use',
 'light_requirements',
 'soil_moisture',
 'soil_ph',
 'soil_description',
 'wildlife_use',
 'interesting_foliage',
 'fragrant_foliage',
 'commercial_availability',
 'propagation_description']

In [62]:
text_columns = ladybird.select_dtypes(include="object").columns

text_columns

Index(['scientific_name', 'common_name', 'usda_code', 'usda_status', 'duration', 'growth_habit', 'size_notes',
       'bloom_color', 'bloom_time', 'distribution', 'water_use', 'light_requirements', 'soil_moisture', 'soil_ph',
       'soil_description', 'wildlife_use', 'interesting_foliage', 'fragrant_foliage', 'commercial_availability',
       'propagation_description'],
      dtype='object')

In [63]:
for column in text_columns:
    ladybird[column] = ladybird[column].apply(clean_text)

In [64]:
ladybird["scientific_name_key"] = (
    ladybird["scientific_name"]
    .apply(normalize_scientific_name)
)

ladybird["usda_code"] = (
    ladybird["usda_code"]
    .apply(normalize_usda_code)
)

ladybird[
    [
        "scientific_name",
        "scientific_name_key",
        "usda_code"
    ]
].head(10)

,scientific_name,scientific_name_key,usda_code
0,Abies amabilis,abies amabilis,ABAM
1,Abies balsamea,abies balsamea,ABBA
2,Abies concolor,abies concolor,ABCO
3,Abies fraseri,abies fraseri,ABFR
4,Abies grandis,abies grandis,ABGR
5,Abies lasiocarpa,abies lasiocarpa,ABLA
6,Abies magnifica,abies magnifica,ABMA
7,Abies procera,abies procera,ABPR
8,Acer circinatum,acer circinatum,ACCI
9,Acer floridanum,acer floridanum,ACFL


In [65]:
duplicate_species = (
    ladybird["scientific_name_key"]
    .value_counts()
)

duplicate_species[
    duplicate_species > 1
].head(20)

Series([], Name: count, dtype: int64)

In [66]:
print("Rows:", len(ladybird))

print(
    "Unique scientific names:",
    ladybird["scientific_name_key"].nunique()
)

print(
    "Unique USDA codes:",
    ladybird["usda_code"].nunique()
)

Rows: 4049
Unique scientific names: 4049
Unique USDA codes: 4049


In [67]:
missing = (
    ladybird.isna()
    .sum()
    .sort_values(ascending=False)
)

missing

fragrant_foliage           3969
interesting_foliage        3732
soil_ph                    3532
wildlife_use               3369
propagation_description    2932
commercial_availability    2794
water_use                  2683
soil_description           2372
soil_moisture              2145
distribution               2100
light_requirements         1794
duration                   1039
size_notes                  267
bloom_time                  168
bloom_color                  30
scientific_name               0
common_name                   0
growth_habit                  0
usda_status                   0
usda_code                     0
scientific_name_key           0
dtype: int64

In [68]:
ladybird["light_requirements"].value_counts(dropna=False)

light_requirements
<NA>                        1794
Sun                          751
Part Shade                   587
Sun , Part Shade , Shade     311
Sun , Part Shade             293
Part Shade , Shade           188
Shade                        114
Sun , Shade                   11
Name: count, dtype: int64

In [69]:
ladybird["water_use"].value_counts(dropna=False)

water_use
<NA>                   2683
Medium                  578
Low                     410
High                    296
Low , Medium             49
Medium , High            31
Low , Medium , High       2
Name: count, dtype: int64

In [70]:
ladybird["growth_habit"].value_counts()

growth_habit
Herb                2395
Shrub                516
Grass/Grass-like     373
Tree                 356
Subshrub             142
Cactus/Succulent      98
Vine                  90
Perennial             29
Biennial              26
Fern                  24
Name: count, dtype: int64

In [71]:
ladybird["bloom_time"].value_counts(dropna=False)

bloom_time
Mar , Apr , May                      233
Jun , Jul , Aug                      231
Apr , May , Jun                      207
May , Jun , Jul , Aug                194
Mar , Apr , May , Jun                193
                                    ... 
Apr , May , Aug                        1
Feb , Mar , May , Jun , Jul , Aug      1
Mar                                    1
Aug                                    1
Mar , Apr , May , Aug , Sep            1
Name: count, Length: 111, dtype: int64

In [72]:
sorted(
    ladybird["light_requirements"]
    .dropna()
    .unique()
)

['Part Shade',
 'Part Shade , Shade',
 'Shade',
 'Sun',
 'Sun , Part Shade',
 'Sun , Part Shade , Shade',
 'Sun , Shade']

In [73]:
def has_value(text, value):
    """
    Returns True if 'value' exists inside a comma-separated string.
    """

    if pd.isna(text):
        return False

    values = [
        item.strip()
        for item in text.split(",")
    ]

    return value in values

In [74]:
ladybird["supports_full_sun"] = (
    ladybird["light_requirements"]
    .apply(lambda x: has_value(x, "Sun"))
)

ladybird["supports_part_shade"] = (
    ladybird["light_requirements"]
    .apply(lambda x: has_value(x, "Part Shade"))
)

ladybird["supports_shade"] = (
    ladybird["light_requirements"]
    .apply(lambda x: has_value(x, "Shade"))
)

In [75]:
ladybird[
    [
        "light_requirements",
        "supports_full_sun",
        "supports_part_shade",
        "supports_shade"
    ]
].sample(10, random_state=42)

,light_requirements,supports_full_sun,supports_part_shade,supports_shade
1159,"Sun , Part Shade , Shade",True,True,True
149,"Part Shade , Shade",False,True,True
3909,Part Shade,False,True,False
3523,<NA>,False,False,False
3206,"Sun , Part Shade , Shade",True,True,True
3985,<NA>,False,False,False
1745,<NA>,False,False,False
2022,Sun,True,False,False
2826,Sun,True,False,False
1042,Part Shade,False,True,False


In [76]:
ladybird["supports_low_water"] = (
    ladybird["water_use"]
    .apply(lambda x: has_value(x, "Low"))
)

ladybird["supports_medium_water"] = (
    ladybird["water_use"]
    .apply(lambda x: has_value(x, "Medium"))
)

ladybird["supports_high_water"] = (
    ladybird["water_use"]
    .apply(lambda x: has_value(x, "High"))
)
ladybird[
    [
        "water_use",
        "supports_low_water",
        "supports_medium_water",
        "supports_high_water"
    ]
].sample(10, random_state=42)

,water_use,supports_low_water,supports_medium_water,supports_high_water
1159,Medium,False,True,False
149,Medium,False,True,False
3909,Low,True,False,False
3523,<NA>,False,False,False
3206,<NA>,False,False,False
3985,<NA>,False,False,False
1745,<NA>,False,False,False
2022,Low,True,False,False
2826,<NA>,False,False,False
1042,Low,True,False,False


In [77]:
MONTHS = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec"
]

for month in MONTHS:

    column = f"bloom_{month.lower()}"

    ladybird[column] = (
        ladybird["bloom_time"]
        .apply(lambda x: has_value(x, month))
    )

bloom_columns = [
    f"bloom_{month.lower()}"
    for month in MONTHS
]

ladybird[
    ["bloom_time"] + bloom_columns
].head()

,bloom_time,bloom_jan,bloom_feb,bloom_mar,bloom_apr,bloom_may,bloom_jun,bloom_jul,bloom_aug,bloom_sep,bloom_oct,bloom_nov,bloom_dec
0,"Apr , May , Jun",False,False,False,True,True,True,False,False,False,False,False,False
1,"Sep , Oct , Nov",False,False,False,False,False,False,False,False,True,True,True,False
2,"Apr , May , Jun",False,False,False,True,True,True,False,False,False,False,False,False
3,Apr,False,False,False,True,False,False,False,False,False,False,False,False
4,"Apr , May",False,False,False,True,True,False,False,False,False,False,False,False


In [78]:
ladybird[
    bloom_columns
].sum().sort_values(ascending=False)

bloom_jun    2800
bloom_may    2562
bloom_jul    2501
bloom_aug    2179
bloom_apr    1935
bloom_sep    1501
bloom_mar    1183
bloom_oct     950
bloom_nov     523
bloom_feb     375
bloom_dec     215
bloom_jan     212
dtype: int64

In [79]:
ladybird[
    [
        "supports_low_water",
        "supports_medium_water",
        "supports_high_water"
    ]
].sum()

supports_low_water       461
supports_medium_water    660
supports_high_water      329
dtype: int64

In [80]:
ladybird.to_pickle(
    PROCESSED_DIR /
    "checkpoint_ladybird_features.pkl"
)

print("Feature-engineered Ladybird dataset saved.")

Feature-engineered Ladybird dataset saved.


## USDA Plant Taxonomy Cleaning

The USDA plant list is used as the reference source for plant symbols, accepted scientific names, synonyms, common names, and plant families.

In [81]:
usda = usda_raw.copy()

print("USDA shape:", usda.shape)
usda.head()

USDA shape: (93157, 5)


,Symbol,Synonym Symbol,Scientific Name with Author,Common Name,Family
0,ABAB,NaN,Abutilon abutiloides (Jacq.) Garcke ex Hochr.,shrubby Indian mallow,Malvaceae
1,ABAB,ABAM5,Abutilon americanum (L.) Sweet,NaN,NaN
2,ABAB,ABJA,Abutilon jacquinii G. Don,NaN,NaN
3,ABAB,ABLI,Abutilon lignosum (Cav.) G. Don,NaN,NaN
4,ABAB70,NaN,Abietinella abietina (Hedw.) Fleisch.,abietinella moss,Thuidiaceae


In [82]:
usda.columns.tolist()

['Symbol',
 'Synonym Symbol',
 'Scientific Name with Author',
 'Common Name',
 'Family']

In [83]:
usda = usda.rename(
    columns={
        "Symbol": "accepted_usda_code",
        "Synonym Symbol": "synonym_usda_code",
        "Scientific Name with Author": "scientific_name_with_author",
        "Common Name": "common_name",
        "Family": "plant_family",
    }
)

usda.columns.tolist()

['accepted_usda_code',
 'synonym_usda_code',
 'scientific_name_with_author',
 'common_name',
 'plant_family']

In [84]:
usda_text_columns = usda.select_dtypes(
    include="object"
).columns

for column in usda_text_columns:
    usda[column] = usda[column].apply(clean_text)

In [85]:
usda["accepted_usda_code"] = (
    usda["accepted_usda_code"]
    .apply(normalize_usda_code)
)

usda["synonym_usda_code"] = (
    usda["synonym_usda_code"]
    .apply(normalize_usda_code)
)

In [86]:
usda[
    [
        "accepted_usda_code",
        "synonym_usda_code",
        "scientific_name_with_author",
        "common_name",
        "plant_family",
    ]
].head(15)

,accepted_usda_code,synonym_usda_code,scientific_name_with_author,common_name,plant_family
0,ABAB,<NA>,Abutilon abutiloides (Jacq.) Garcke ex Hochr.,shrubby Indian mallow,Malvaceae
1,ABAB,ABAM5,Abutilon americanum (L.) Sweet,<NA>,<NA>
2,ABAB,ABJA,Abutilon jacquinii G. Don,<NA>,<NA>
3,ABAB,ABLI,Abutilon lignosum (Cav.) G. Don,<NA>,<NA>
4,ABAB70,<NA>,Abietinella abietina (Hedw.) Fleisch.,abietinella moss,Thuidiaceae
5,ABAB70,HYAB,Hypnum abietinum Hedw.,<NA>,<NA>
6,ABAB70,THAB70,Thuidium abietinum (Hedw.) Schimp.,<NA>,<NA>
7,ABAL,<NA>,Abronia alpina Brandegee,Ramshaw Meadows sand verbena,Nyctaginaceae
8,ABAL3,<NA>,Abies alba Mill.,silver fir,Pinaceae
9,ABAM,<NA>,Abies amabilis (Douglas ex Loudon) Douglas ex Forbes,Pacific silver fir,Pinaceae


In [87]:
accepted_row_count = (
    usda["synonym_usda_code"]
    .isna()
    .sum()
)

synonym_row_count = (
    usda["synonym_usda_code"]
    .notna()
    .sum()
)

print("Accepted rows:", accepted_row_count)
print("Synonym rows:", synonym_row_count)
print("Total rows:", len(usda))

Accepted rows: 48994
Synonym rows: 44163
Total rows: 93157


In [88]:
usda_accepted = (
    usda.loc[
        usda["synonym_usda_code"].isna()
    ]
    .copy()
)

print("Accepted USDA records:", len(usda_accepted))

usda_accepted.head()

Accepted USDA records: 48994


,accepted_usda_code,synonym_usda_code,scientific_name_with_author,common_name,plant_family
0,ABAB,<NA>,Abutilon abutiloides (Jacq.) Garcke ex Hochr.,shrubby Indian mallow,Malvaceae
4,ABAB70,<NA>,Abietinella abietina (Hedw.) Fleisch.,abietinella moss,Thuidiaceae
7,ABAL,<NA>,Abronia alpina Brandegee,Ramshaw Meadows sand verbena,Nyctaginaceae
8,ABAL3,<NA>,Abies alba Mill.,silver fir,Pinaceae
9,ABAM,<NA>,Abies amabilis (Douglas ex Loudon) Douglas ex Forbes,Pacific silver fir,Pinaceae


In [89]:
usda_accepted = usda_accepted.drop(
    columns="synonym_usda_code"
)

In [90]:
accepted_code_duplicates = (
    usda_accepted["accepted_usda_code"]
    .value_counts()
)

accepted_code_duplicates[
    accepted_code_duplicates > 1
].head(20)

Series([], Name: count, dtype: int64)

In [91]:
print(
    "Accepted rows:",
    len(usda_accepted),
)

print(
    "Unique accepted USDA codes:",
    usda_accepted[
        "accepted_usda_code"
    ].nunique(),
)

Accepted rows: 48994
Unique accepted USDA codes: 48994


In [92]:
usda_synonyms = (
    usda.loc[
        usda["synonym_usda_code"].notna(),
        [
            "synonym_usda_code",
            "accepted_usda_code",
            "scientific_name_with_author",
        ],
    ]
    .copy()
)

print("Synonym mappings:", len(usda_synonyms))

usda_synonyms.head()

Synonym mappings: 44163


,synonym_usda_code,accepted_usda_code,scientific_name_with_author
1,ABAM5,ABAB,Abutilon americanum (L.) Sweet
2,ABJA,ABAB,Abutilon jacquinii G. Don
3,ABLI,ABAB,Abutilon lignosum (Cav.) G. Don
5,HYAB,ABAB70,Hypnum abietinum Hedw.
6,THAB70,ABAB70,Thuidium abietinum (Hedw.) Schimp.


In [93]:
synonym_conflicts = (
    usda_synonyms.groupby(
        "synonym_usda_code"
    )["accepted_usda_code"]
    .nunique()
)

synonym_conflicts[
    synonym_conflicts > 1
].head(20)

Series([], Name: accepted_usda_code, dtype: int64)

In [94]:
accepted_code_crosswalk = (
    usda_accepted[
        ["accepted_usda_code"]
    ]
    .copy()
)

accepted_code_crosswalk[
    "source_usda_code"
] = accepted_code_crosswalk[
    "accepted_usda_code"
]

accepted_code_crosswalk[
    "code_type"
] = "accepted"

In [95]:
synonym_code_crosswalk = (
    usda_synonyms[
        [
            "synonym_usda_code",
            "accepted_usda_code",
        ]
    ]
    .rename(
        columns={
            "synonym_usda_code":
                "source_usda_code"
        }
    )
    .copy()
)

synonym_code_crosswalk[
    "code_type"
] = "synonym"

In [96]:
synonym_code_crosswalk = (
    usda_synonyms[
        [
            "synonym_usda_code",
            "accepted_usda_code",
        ]
    ]
    .rename(
        columns={
            "synonym_usda_code":
                "source_usda_code"
        }
    )
    .copy()
)

synonym_code_crosswalk[
    "code_type"
] = "synonym"

In [97]:
usda_code_crosswalk = pd.concat(
    [
        accepted_code_crosswalk[
            [
                "source_usda_code",
                "accepted_usda_code",
                "code_type",
            ]
        ],
        synonym_code_crosswalk[
            [
                "source_usda_code",
                "accepted_usda_code",
                "code_type",
            ]
        ],
    ],
    ignore_index=True,
)

usda_code_crosswalk.head(10)

,source_usda_code,accepted_usda_code,code_type
0,ABAB,ABAB,accepted
1,ABAB70,ABAB70,accepted
2,ABAL,ABAL,accepted
3,ABAL3,ABAL3,accepted
4,ABAM,ABAM,accepted
5,ABAM2,ABAM2,accepted
6,ABAM3,ABAM3,accepted
7,ABAN,ABAN,accepted
8,ABAR,ABAR,accepted
9,ABAU,ABAU,accepted


In [98]:
crosswalk_duplicates = (
    usda_code_crosswalk[
        "source_usda_code"
    ]
    .value_counts()
)

crosswalk_duplicates[
    crosswalk_duplicates > 1
].head(20)

source_usda_code
HEST9     2
SCHY5     2
SPMAN     2
EUCH14    2
GLTE6     2
SCLA12    2
TECUD2    2
CABR42    2
MAPA20    2
MAHY4     2
CLTH      2
MAPA21    2
ALTI2     2
Name: count, dtype: int64

In [99]:
TAXONOMIC_RANKS = {
    "subsp.",
    "ssp.",
    "var.",
    "f.",
}

In [100]:
def extract_taxonomic_name(name):
    """
    Remove botanical-author information while preserving
    species, subspecies, variety, and form names where possible.
    """

    name = clean_text(name)

    if pd.isna(name):
        return pd.NA

    words = name.split()

    if len(words) < 2:
        return name

    result = words[:2]

    for index, word in enumerate(words[2:], start=2):
        if word in TAXONOMIC_RANKS:
            if index + 1 < len(words):
                result.extend(
                    [
                        word,
                        words[index + 1],
                    ]
                )
            break

    return " ".join(result)

In [101]:
test_names = [
    "Abies balsamea (L.) Mill.",
    "Acer glabrum Torr. var. douglasii (Hook.) Dippel",
    "Alnus incana (L.) Moench subsp. rugosa (Du Roi) R.T. Clausen",
    "Quercus alba L.",
]

for name in test_names:
    print(
        name,
        " --> ",
        extract_taxonomic_name(name),
    )

Abies balsamea (L.) Mill.  -->  Abies balsamea
Acer glabrum Torr. var. douglasii (Hook.) Dippel  -->  Acer glabrum var. douglasii
Alnus incana (L.) Moench subsp. rugosa (Du Roi) R.T. Clausen  -->  Alnus incana subsp. rugosa
Quercus alba L.  -->  Quercus alba


In [102]:
usda_accepted[
    "scientific_name"
] = (
    usda_accepted[
        "scientific_name_with_author"
    ]
    .apply(extract_taxonomic_name)
)

usda_accepted[
    "scientific_name_key"
] = (
    usda_accepted[
        "scientific_name"
    ]
    .apply(normalize_scientific_name)
)

In [103]:
usda_accepted[
    [
        "accepted_usda_code",
        "scientific_name_with_author",
        "scientific_name",
        "scientific_name_key",
        "common_name",
        "plant_family",
    ]
].head(20)

,accepted_usda_code,scientific_name_with_author,scientific_name,scientific_name_key,common_name,plant_family
0,ABAB,Abutilon abutiloides (Jacq.) Garcke ex Hochr.,Abutilon abutiloides,abutilon abutiloides,shrubby Indian mallow,Malvaceae
4,ABAB70,Abietinella abietina (Hedw.) Fleisch.,Abietinella abietina,abietinella abietina,abietinella moss,Thuidiaceae
7,ABAL,Abronia alpina Brandegee,Abronia alpina,abronia alpina,Ramshaw Meadows sand verbena,Nyctaginaceae
8,ABAL3,Abies alba Mill.,Abies alba,abies alba,silver fir,Pinaceae
9,ABAM,Abies amabilis (Douglas ex Loudon) Douglas ex Forbes,Abies amabilis,abies amabilis,Pacific silver fir,Pinaceae
10,ABAM2,Abronia ameliae Lundell,Abronia ameliae,abronia ameliae,Amelia's sand verbena,Nyctaginaceae
11,ABAM3,Abronia ammophila Greene,Abronia ammophila,abronia ammophila,Wyoming sand verbena,Nyctaginaceae
13,ABAN,Abronia angustifolia Greene,Abronia angustifolia,abronia angustifolia,purple sand verbena,Nyctaginaceae
16,ABAR,Abronia argillosa S.L. Welsh & Goodrich,Abronia argillosa,abronia argillosa,clay sand verbena,Nyctaginaceae
17,ABAU,Abutilon auritum (Wall. ex Link) Sweet,Abutilon auritum,abutilon auritum,Asian Indian mallow,Malvaceae


In [104]:
usda_name_duplicates = (
    usda_accepted[
        "scientific_name_key"
    ]
    .value_counts()
)

usda_name_duplicates[
    usda_name_duplicates > 1
].head(20)

scientific_name_key
ericameria nauseosa ssp. nauseosa              15
monarda punctata ssp. punctata                  9
ericameria nauseosa ssp. consimilis             9
machaeranthera canescens ssp. canescens         8
lupinus lyallii ssp. lyallii                    7
lupinus sericeus ssp. sericeus                  7
cephaloziella rubella ssp. rubella              6
sidalcea oregana ssp. oregana                   6
solidago simplex ssp. randii                    6
silene scouleri ssp. pringlei                   6
chamaecrista nictitans ssp. nictitans           6
lupinus sellulus ssp. sellulus                  5
artemisia campestris ssp. borealis              5
arnica chamissonis ssp. foliosa                 5
monarda fistulosa ssp. fistulosa                5
ptelea trifoliata ssp. pallida                  5
salvia dorrii ssp. dorrii                       5
solidago simplex ssp. simplex                   5
lathyrus nevadensis ssp. lanceolatus            5
symphyotrichum lanceolatum ssp

In [105]:
ladybird_usda_check = (
    ladybird[
        [
            "scientific_name",
            "scientific_name_key",
            "usda_code",
        ]
    ]
    .merge(
        usda_accepted[
            [
                "accepted_usda_code",
                "scientific_name",
                "scientific_name_key",
                "plant_family",
            ]
        ],
        left_on="usda_code",
        right_on="accepted_usda_code",
        how="left",
        suffixes=(
            "_ladybird",
            "_usda",
        ),
        indicator=True,
    )
)

In [106]:
ladybird_usda_check[
    "_merge"
].value_counts()

_merge
both          4048
left_only        1
right_only       0
Name: count, dtype: int64

In [107]:
ladybird_usda_unmatched = (
    ladybird_usda_check.loc[
        ladybird_usda_check["_merge"]
        == "left_only",
        [
            "scientific_name_ladybird",
            "scientific_name_key_ladybird",
            "usda_code",
        ],
    ]
)

print(
    "Unmatched Ladybird USDA codes:",
    len(ladybird_usda_unmatched),
)

ladybird_usda_unmatched.head(20)

Unmatched Ladybird USDA codes: 1


,scientific_name_ladybird,scientific_name_key_ladybird,usda_code
307,Celtis tenuifolia,celtis tenuifolia,CETE


In [108]:
ladybird_code_resolution = (
    ladybird[
        [
            "scientific_name",
            "scientific_name_key",
            "usda_code",
        ]
    ]
    .merge(
        usda_code_crosswalk,
        left_on="usda_code",
        right_on="source_usda_code",
        how="left",
        indicator=True,
    )
)

In [109]:
ladybird_code_resolution[
    "_merge"
].value_counts()

_merge
both          4049
left_only        0
right_only       0
Name: count, dtype: int64

In [110]:
ladybird_code_resolution.loc[
    ladybird_code_resolution[
        "code_type"
    ]
    == "synonym",
    [
        "scientific_name",
        "usda_code",
        "accepted_usda_code",
        "code_type",
    ],
].head(20)

,scientific_name,usda_code,accepted_usda_code,code_type
307,Celtis tenuifolia,CETE,CEPU10,synonym


In [111]:
usda_accepted.to_pickle(
    PROCESSED_DIR
    / "checkpoint_usda_accepted.pkl"
)

usda_synonyms.to_pickle(
    PROCESSED_DIR
    / "checkpoint_usda_synonyms.pkl"
)

usda_code_crosswalk.to_pickle(
    PROCESSED_DIR
    / "checkpoint_usda_code_crosswalk.pkl"
)

print("USDA taxonomy checkpoints saved.")

USDA taxonomy checkpoints saved.


In [113]:
ambiguous_crosswalk_codes = (
    usda_code_crosswalk[
        usda_code_crosswalk.duplicated(
            subset="source_usda_code",
            keep=False,
        )
    ]
    .sort_values(
        [
            "source_usda_code",
            "code_type",
        ]
    )
)

print(
    "Ambiguous source codes:",
    ambiguous_crosswalk_codes[
        "source_usda_code"
    ].nunique(),
)

ambiguous_crosswalk_codes

Ambiguous source codes: 13


,source_usda_code,accepted_usda_code,code_type
1634,ALTI2,ALTI2,accepted
50266,ALTI2,ALTI3,synonym
7172,CABR42,CABR42,accepted
54906,CABR42,CABR18,synonym
11424,CLTH,CLTH,accepted
58526,CLTH,CLTH5,synonym
18511,EUCH14,EUCH14,accepted
57140,EUCH14,CHCH16,synonym
20777,GLTE6,GLTE6,accepted
66881,GLTE6,GLBI3,synonym


In [114]:
ladybird_ambiguous_codes = (
    ladybird[
        ladybird["usda_code"].isin(
            ambiguous_crosswalk_codes[
                "source_usda_code"
            ]
        )
    ][
        [
            "scientific_name",
            "usda_code",
        ]
    ]
)

print(
    "Ladybird plants using ambiguous codes:",
    len(ladybird_ambiguous_codes),
)

ladybird_ambiguous_codes

Ladybird plants using ambiguous codes: 0


,scientific_name,usda_code


In [115]:
usda_code_crosswalk_unique = (
    usda_code_crosswalk
    .assign(
        code_priority=(
            usda_code_crosswalk[
                "code_type"
            ]
            .map(
                {
                    "accepted": 1,
                    "synonym": 2,
                }
            )
        )
    )
    .sort_values(
        [
            "source_usda_code",
            "code_priority",
        ]
    )
    .drop_duplicates(
        subset="source_usda_code",
        keep="first",
    )
    .drop(columns="code_priority")
)

print(
    "Rows in original crosswalk:",
    len(usda_code_crosswalk),
)

print(
    "Rows in unique crosswalk:",
    len(usda_code_crosswalk_unique),
)

print(
    "Duplicate source codes remaining:",
    usda_code_crosswalk_unique[
        "source_usda_code"
    ].duplicated().sum(),
)

Rows in original crosswalk: 93157
Rows in unique crosswalk: 93144
Duplicate source codes remaining: 0


In [116]:
ladybird = ladybird.merge(
    usda_code_crosswalk_unique,
    left_on="usda_code",
    right_on="source_usda_code",
    how="left",
    validate="one_to_one",
)

In [117]:
print("Rows after merge:", len(ladybird))

print(
    "Missing accepted USDA codes:",
    ladybird["accepted_usda_code"].isna().sum(),
)

Rows after merge: 4049
Missing accepted USDA codes: 0


In [118]:
ladybird[
    [
        "scientific_name",
        "usda_code",
        "accepted_usda_code",
        "code_type",
    ]
].head(10)

,scientific_name,usda_code,accepted_usda_code,code_type
0,Abies amabilis,ABAM,ABAM,accepted
1,Abies balsamea,ABBA,ABBA,accepted
2,Abies concolor,ABCO,ABCO,accepted
3,Abies fraseri,ABFR,ABFR,accepted
4,Abies grandis,ABGR,ABGR,accepted
5,Abies lasiocarpa,ABLA,ABLA,accepted
6,Abies magnifica,ABMA,ABMA,accepted
7,Abies procera,ABPR,ABPR,accepted
8,Acer circinatum,ACCI,ACCI,accepted
9,Acer floridanum,ACFL,ACFL,accepted


In [119]:
ladybird = ladybird.merge(
    usda_accepted[
        [
            "accepted_usda_code",
            "scientific_name_with_author",
            "scientific_name",
            "scientific_name_key",
            "common_name",
            "plant_family",
        ]
    ],
    on="accepted_usda_code",
    how="left",
    suffixes=("_ladybird", "_usda"),
    validate="many_to_one",
)

In [120]:
ladybird["preferred_scientific_name"] = (
    ladybird["scientific_name_usda"]
    .combine_first(
        ladybird["scientific_name_ladybird"]
    )
)

ladybird["preferred_scientific_name_key"] = (
    ladybird["scientific_name_key_usda"]
    .combine_first(
        ladybird["scientific_name_key_ladybird"]
    )
)

ladybird["preferred_common_name"] = (
    ladybird["common_name_ladybird"]
    .combine_first(
        ladybird["common_name_usda"]
    )
)

In [121]:
ladybird.loc[
    ladybird["code_type"] == "synonym",
    [
        "scientific_name_ladybird",
        "usda_code",
        "accepted_usda_code",
        "scientific_name_usda",
        "preferred_scientific_name",
        "plant_family",
    ],
]

,scientific_name_ladybird,usda_code,accepted_usda_code,scientific_name_usda,preferred_scientific_name,plant_family
307,Celtis tenuifolia,CETE,CEPU10,Celtis pumila,Celtis pumila,Ulmaceae


In [122]:
print("Rows:", len(ladybird))

print(
    "Unique accepted USDA codes:",
    ladybird["accepted_usda_code"].nunique(),
)

print(
    "Missing plant families:",
    ladybird["plant_family"].isna().sum(),
)

print(
    "Missing preferred scientific names:",
    ladybird["preferred_scientific_name"].isna().sum(),
)

Rows: 4049
Unique accepted USDA codes: 4048
Missing plant families: 0
Missing preferred scientific names: 0


In [123]:
ladybird[
    [
        "preferred_scientific_name",
        "preferred_common_name",
        "accepted_usda_code",
        "plant_family",
        "growth_habit",
        "light_requirements",
        "water_use",
    ]
].head(10)

,preferred_scientific_name,preferred_common_name,accepted_usda_code,plant_family,growth_habit,light_requirements,water_use
0,Abies amabilis,"Pacific Silver Fir, Cascade Fir, Lovely Fir, White Fir, Red fir",ABAM,Pinaceae,Tree,Shade,Medium
1,Abies balsamea,"Balsam Fir, Blister Pine, Northern Balsam",ABBA,Pinaceae,Tree,"Sun , Part Shade , Shade",Medium
2,Abies concolor,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",ABCO,Pinaceae,Tree,"Sun , Part Shade",Medium
3,Abies fraseri,"Fraser Fir, She-balsam",ABFR,Pinaceae,Tree,Shade,Medium
4,Abies grandis,"Grand Fir, Giant Fir",ABGR,Pinaceae,Tree,"Part Shade , Shade",Medium
5,Abies lasiocarpa,Subalpine Fir,ABLA,Pinaceae,Tree,"Sun , Part Shade , Shade",Medium
6,Abies magnifica,"California Red Fir, Red Fir",ABMA,Pinaceae,Tree,Sun,Medium
7,Abies procera,Noble Fir,ABPR,Pinaceae,Tree,Sun,Low
8,Acer circinatum,"Oregon Vine Maple, Vine Maple",ACCI,Aceraceae,Shrub,"Part Shade , Shade",Medium
9,Acer floridanum,"Southern Sugar Maple, Florida Maple, Caddo Maple, Rock Maple",ACFL,Aceraceae,Tree,Part Shade,Medium


In [124]:
ladybird.to_pickle(
    PROCESSED_DIR
    / "checkpoint_ladybird_usda_enriched.pkl"
)

print("USDA-enriched Ladybird checkpoint saved.")

USDA-enriched Ladybird checkpoint saved.


## Pollinator Interaction Data

The pollinator dataset contains observation-level records. It must be cleaned and aggregated so that each plant has one summary row before merging into the master dataset.

In [125]:
pollinator = pollinator_raw.copy()

print("Pollinator shape:", pollinator.shape)

pollinator.head()

Pollinator shape: (67954, 23)


,catalog_number,pollinator_family,pollinator_genus,pollinator_species,pollinator_sex,plant_sp_code,plant_species,collection_method,pantrap_colour,collector_number,day_collected,month_collected,year_collected,time_of_day_collected,location_description,location_name,habitat,latitude,longitude,basis_of_record,location_specimen,study_director,study
0,SFU705915,Tachinidae,Tachinidae,NaN,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
1,SFU705916,Halictidae,Lasioglossum,sp. 1,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
2,SFU705917,Andrenidae,Andrena,nigrocaerulea,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
3,SFU705918,Sarcophagidae,Sarcophagidae,NaN,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007
4,SFU705919,Halictidae,Lasioglossum,incompletum,F,NaN,NaN,"Pantrap, blue",blue,20,30,Apr,2007,NaN,"Bear Hill Regional Park, Victoria CRD",BH,Garry Oak,48.545605,-123.406769,specimen,SFU,Neame,GO_PT_2007


In [126]:
pollinator.columns.tolist()

['catalog_number',
 'pollinator_family',
 'pollinator_genus',
 'pollinator_species',
 'pollinator_sex',
 'plant_sp_code',
 'plant_species',
 'collection_method',
 'pantrap_colour',
 'collector_number',
 'day_collected',
 'month_collected',
 'year_collected',
 'time_of_day_collected',
 'location_description',
 'location_name',
 'habitat',
 'latitude',
 'longitude',
 'basis_of_record',
 'location_specimen',
 'study_director',
 'study']

In [127]:
pollinator_profile = pd.DataFrame(
    {
        "column": pollinator.columns,
        "data_type": pollinator.dtypes.astype(str).values,
        "missing_count": pollinator.isna().sum().values,
    }
)

pollinator_profile["missing_percent"] = (
    pollinator_profile["missing_count"]
    / len(pollinator)
    * 100
).round(2)

pollinator_profile

,column,data_type,missing_count,missing_percent
0,catalog_number,object,0,0.00
1,pollinator_family,object,8,0.01
2,pollinator_genus,object,0,0.00
3,pollinator_species,object,6904,10.16
4,pollinator_sex,object,12299,18.10
5,plant_sp_code,object,33005,48.57
6,plant_species,object,33005,48.57
7,collection_method,object,0,0.00
8,pantrap_colour,object,41535,61.12
9,collector_number,int64,0,0.00


In [128]:
def normalize_column_name(column):
    column = clean_text(column)

    if pd.isna(column):
        return column

    column = column.casefold()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = column.strip("_")

    return column

In [129]:
pollinator.columns = [
    normalize_column_name(column)
    for column in pollinator.columns
]

pollinator.columns.tolist()

['catalog_number',
 'pollinator_family',
 'pollinator_genus',
 'pollinator_species',
 'pollinator_sex',
 'plant_sp_code',
 'plant_species',
 'collection_method',
 'pantrap_colour',
 'collector_number',
 'day_collected',
 'month_collected',
 'year_collected',
 'time_of_day_collected',
 'location_description',
 'location_name',
 'habitat',
 'latitude',
 'longitude',
 'basis_of_record',
 'location_specimen',
 'study_director',
 'study']

In [130]:
required_pollinator_columns = [
    "plant_species",
    "pollinator_family",
    "pollinator_genus",
    "pollinator_species",
]

missing_required_columns = [
    column
    for column in required_pollinator_columns
    if column not in pollinator.columns
]

if missing_required_columns:
    print(
        "Missing required columns:",
        missing_required_columns,
    )
else:
    print("All required pollinator columns were found.")

All required pollinator columns were found.


In [131]:
pollinator_text_columns = (
    pollinator.select_dtypes(
        include="object"
    ).columns
)

for column in pollinator_text_columns:
    pollinator[column] = (
        pollinator[column]
        .apply(clean_text)
    )

In [132]:
pollinator["scientific_name_key"] = (
    pollinator["plant_species"]
    .apply(normalize_scientific_name)
)

In [133]:
pollinator[
    [
        "plant_species",
        "scientific_name_key",
    ]
].head(10)

,plant_species,scientific_name_key
0,<NA>,<NA>
1,<NA>,<NA>
2,<NA>,<NA>
3,<NA>,<NA>
4,<NA>,<NA>
5,<NA>,<NA>
6,<NA>,<NA>
7,<NA>,<NA>
8,<NA>,<NA>
9,<NA>,<NA>


In [134]:
print("Total pollinator records:", len(pollinator))

print(
    "Records with plant names:",
    pollinator["scientific_name_key"]
    .notna()
    .sum(),
)

print(
    "Records without plant names:",
    pollinator["scientific_name_key"]
    .isna()
    .sum(),
)

print(
    "Unique plant names:",
    pollinator["scientific_name_key"]
    .nunique(),
)

Total pollinator records: 67954
Records with plant names: 34949
Records without plant names: 33005
Unique plant names: 472


In [135]:
pollinator_identified = (
    pollinator.loc[
        pollinator[
            "scientific_name_key"
        ].notna()
    ]
    .copy()
)

print(
    "Identified plant records:",
    len(pollinator_identified),
)

Identified plant records: 34949


In [136]:
pollinator_identified[
    "pollinator_family"
].value_counts(
    dropna=False
).head(30)

pollinator_family
Apidae           14926
Halictidae        6388
Andrenidae        3513
Syrphidae         3128
Megachilidae      2180
Bombylidae         762
Empididae          410
Muscoidea          304
Cerambycidae       293
Tachinidae         293
Vespidae           253
Buprestidae        170
Anthomyiidae       166
Colletidae         146
Adelidae           142
Muscidae           125
Bruchidae          113
Diptera            101
Lycaenidae         100
Crabronidae         95
Sarcophagidae       89
Calliphoridae       83
Conopidae           80
Lygaeidae           69
Elateridae          68
Sphecidae           59
Mordellidae         58
Ichneumonidae       55
Cimbicidae          55
Coleoptera          52
Name: count, dtype: int64

In [137]:
pollinator_identified[
    "pollinator_genus"
].value_counts(
    dropna=False
).head(30)

pollinator_genus
Bombus              8357
Lasioglossum        4899
Apis                4874
Andrena             3423
Osmia               1618
Halictus            1223
Ceratina            1111
Bombylius            587
Volucella            529
Eristalis            470
Toxomerus            427
Muscoidea            304
Sphaerophoria        303
Nomada               287
Empididae            262
Eupeodes             248
Merodon\nMerodon     223
Cortodera            218
Megachile            214
Platycheirus         191
Agapostemon          184
Anthidium            176
Anthomyiidae         166
Conophorus           149
Adela                142
Melanostoma          140
Buprestidae          135
Tachinidae           124
Rhamphomyia          117
Bruchidae            113
Name: count, dtype: int64

In [138]:
pollinator_identified[
    [
        "pollinator_family",
        "pollinator_genus",
        "pollinator_species",
    ]
].isna().sum()

pollinator_family        5
pollinator_genus         0
pollinator_species    1577
dtype: int64

In [139]:
BEE_FAMILIES = {
    "Apidae",
    "Andrenidae",
    "Colletidae",
    "Halictidae",
    "Megachilidae",
    "Melittidae",
}

BUTTERFLY_FAMILIES = {
    "Papilionidae",
    "Pieridae",
    "Nymphalidae",
    "Lycaenidae",
    "Hesperiidae",
    "Riodinidae",
}

MOTH_FAMILIES = {
    "Sphingidae",
    "Noctuidae",
    "Geometridae",
    "Erebidae",
    "Crambidae",
    "Tortricidae",
}

FLY_FAMILIES = {
    "Syrphidae",
    "Bombyliidae",
    "Tachinidae",
    "Muscidae",
    "Anthomyiidae",
    "Calliphoridae",
}

BEETLE_FAMILIES = {
    "Scarabaeidae",
    "Cerambycidae",
    "Chrysomelidae",
    "Mordellidae",
    "Nitidulidae",
}

WASP_FAMILIES = {
    "Vespidae",
    "Sphecidae",
    "Crabronidae",
    "Pompilidae",
    "Scoliidae",
}

In [140]:
def classify_pollinator_group(row):
    family = row.get("pollinator_family")
    genus = row.get("pollinator_genus")
    species = row.get("pollinator_species")

    combined_text = " ".join(
        str(value)
        for value in [family, genus, species]
        if pd.notna(value)
    ).casefold()

    if family in BEE_FAMILIES:
        return "Bee"

    if family in BUTTERFLY_FAMILIES:
        return "Butterfly"

    if family in MOTH_FAMILIES:
        return "Moth"

    if family in FLY_FAMILIES:
        return "Fly"

    if family in BEETLE_FAMILIES:
        return "Beetle"

    if family in WASP_FAMILIES:
        return "Wasp"

    if any(
        term in combined_text
        for term in [
            "hummingbird",
            "trochilidae",
        ]
    ):
        return "Hummingbird"

    return "Other or Unclassified"

In [141]:
pollinator_identified[
    "pollinator_group"
] = (
    pollinator_identified.apply(
        classify_pollinator_group,
        axis=1,
    )
)

In [142]:
pollinator_identified[
    "pollinator_group"
].value_counts(
    dropna=False
)

pollinator_group
Bee                      27153
Fly                       3836
Other or Unclassified     2948
Wasp                       444
Beetle                     369
Butterfly                  174
Hummingbird                 19
Moth                         6
Name: count, dtype: int64

In [143]:
unclassified_families = (
    pollinator_identified.loc[
        pollinator_identified[
            "pollinator_group"
        ]
        == "Other or Unclassified",
        "pollinator_family",
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "pollinator_family"
    )
    .reset_index(
        name="observation_count"
    )
)

unclassified_families.head(30)

,pollinator_family,observation_count
0,Bombylidae,762
1,Empididae,410
2,Muscoidea,304
3,Buprestidae,170
4,Adelidae,142
5,Bruchidae,113
6,Diptera,101
7,Sarcophagidae,89
8,Conopidae,80
9,Lygaeidae,69


In [144]:
unclassified_families.to_csv(
    REVIEW_DIR
    / "unclassified_pollinator_families.csv",
    index=False,
)

In [145]:
def combine_unique_values(series):
    values = sorted(
        {
            str(value).strip()
            for value in series.dropna()
            if str(value).strip()
        }
    )

    if not values:
        return pd.NA

    return " | ".join(values)

In [146]:
possible_id_columns = [
    "catalog_number",
    "record_id",
    "observation_id",
]

observation_id_column = next(
    (
        column
        for column in possible_id_columns
        if column in pollinator_identified.columns
    ),
    None,
)

print(
    "Observation ID column:",
    observation_id_column,
)

Observation ID column: catalog_number


In [147]:
aggregation_rules = {
    "plant_species": "first",
    "pollinator_family": combine_unique_values,
    "pollinator_genus": combine_unique_values,
    "pollinator_species": combine_unique_values,
    "pollinator_group": combine_unique_values,
}

In [148]:
for optional_column in [
    "habitat",
    "location_name",
    "study",
]:
    if optional_column in pollinator_identified.columns:
        aggregation_rules[
            optional_column
        ] = combine_unique_values

In [149]:
pollinator_summary = (
    pollinator_identified.groupby(
        "scientific_name_key",
        as_index=False,
    )
    .agg(aggregation_rules)
)

In [150]:
if observation_id_column is not None:
    observation_counts = (
        pollinator_identified.groupby(
            "scientific_name_key"
        )[observation_id_column]
        .agg(
            pollinator_observation_count="size",
            unique_observation_count="nunique",
        )
        .reset_index()
    )
else:
    observation_counts = (
        pollinator_identified.groupby(
            "scientific_name_key"
        )
        .size()
        .rename(
            "pollinator_observation_count"
        )
        .reset_index()
    )

    observation_counts[
        "unique_observation_count"
    ] = observation_counts[
        "pollinator_observation_count"
    ]

In [151]:
pollinator_summary = (
    pollinator_summary.merge(
        observation_counts,
        on="scientific_name_key",
        how="left",
        validate="one_to_one",
    )
)

In [152]:
print(
    "Rows in plant-level pollinator summary:",
    len(pollinator_summary),
)

pollinator_summary.head()

Rows in plant-level pollinator summary: 472


,scientific_name_key,plant_species,pollinator_family,pollinator_genus,pollinator_species,pollinator_group,habitat,location_name,study,pollinator_observation_count,unique_observation_count
0,abelia,Abelia,Apidae | Halictidae,Apis | Bombus | Halictus,californicus | mellifera | mixtus | rubicundus | vosnesenskii,Bee,Garry Oak and Associated Ecosystems,MT-U | UP-U | VC | XH-U,OS_NT_2012,7,7
1,achillea,Achillea,Apidae | Colletidae | Crabronidae | Halictidae | Megachilidae | Muscidae | Syrphidae,Agapostemon | Bembix | Bombus | Ceratina | Colletes | Eristalis | Halictus | Heriades | Lasioglossum | Melissodes | Muscidae | Philanthus | Syritt...,acantha | arbustorum | carinatus | confusus | cressonii | fulgidus | microsticta | occidentalis | opinator | pipiens | sp. | texanus | vosnesenskii,Bee | Fly | Wasp,Garry Oak and Associated Ecosystems | Hedgerow Restoration,BB | BBB-T | CB | HC-U | MT-U | OM-C | OM-T | XH,HR_NT_2013 | OS_NT_2012,23,23
2,achillea millefolium,Achillea millefolium,Adelidae | Andrenidae | Apidae | Bibionidae | Bombylidae | Bombyliidae | Bruchidae | Buprestidae | Calliphoridae | Cerambycidae | Chrysididae | Ci...,Adela | Agapostemon | Ammophila | Anastrangalia | Andrena | Anthidium | Apis | Bembix | Bibio | Bibionidae | Bombus | Bombylius | Bruchidae | Bupr...,"""albosignata"" | (Nomada) sp. 3 | abundipunctum | acantha | albipenne | americana | amoena | amphibola | arbustorum | bifarius | buckelli | califor...",Bee | Beetle | Butterfly | Fly | Other or Unclassified | Wasp,Garry Oak | Garry Oak and Associated Ecosystems | Hedgerow Restoration | Highbush Blueberry Crop | Pasture | Scrubbsteppe,AH | Ab2 | BBB-T | BBO-C | BH | CB | CR | FS | GH | GO | HC | HC-U | HLG | HLU | JF | KN | KP | LLa | LP | MH | MM | MO | MT | MZ | OH | OKG | OKU...,GO_NT_2009_2010 | GO_NT_2014_2015 | HB_NT_2016 | HR_NT_2013 | OS_NT_2012 | OS_NT_2017 | PS_NT_2016 | SC_NT_2010,748,748
3,achlys triphylla,Achlys triphylla,Syrphidae,Melanostoma | Parasyrphus,insolitus | mellinum,Fly,Garry Oak and Associated Ecosystems,OH-F | TL-F,OS_NT_2012,2,2
4,aegopodium,Aegopodium,Syrphidae,Syritta | Toxomerus,occidentalis | pipiens,Fly,Garry Oak and Associated Ecosystems,BB | VC,OS_NT_2012,2,2


In [153]:
def contains_group(value, group):
    if pd.isna(value):
        return False

    groups = {
        item.strip()
        for item in str(value).split("|")
    }

    return group in groups

In [154]:
POLLINATOR_GROUPS = [
    "Bee",
    "Butterfly",
    "Moth",
    "Fly",
    "Beetle",
    "Wasp",
    "Hummingbird",
]

for group in POLLINATOR_GROUPS:
    column_name = (
        "supports_"
        + group.casefold()
        + "_observed"
    )

    pollinator_summary[
        column_name
    ] = (
        pollinator_summary[
            "pollinator_group"
        ]
        .apply(
            lambda value: contains_group(
                value,
                group,
            )
        )
    )

In [155]:
pollinator_boolean_columns = [
    column
    for column in pollinator_summary.columns
    if column.startswith("supports_")
]

pollinator_summary[
    [
        "plant_species",
        "pollinator_group",
    ]
    + pollinator_boolean_columns
].sample(
    n=min(
        10,
        len(pollinator_summary),
    ),
    random_state=42,
)

,plant_species,pollinator_group,supports_bee_observed,supports_butterfly_observed,supports_moth_observed,supports_fly_observed,supports_beetle_observed,supports_wasp_observed,supports_hummingbird_observed
55,Calochortus tolmiei,Bee | Butterfly | Fly,True,True,False,True,False,False,False
73,Cerastium arvense,Bee | Fly | Other or Unclassified,True,False,False,True,False,False,False
33,Aster,Bee | Fly | Other or Unclassified | Wasp,True,False,False,True,False,True,False
278,Malus fusca,Bee,True,False,False,False,False,False,False
244,"Lithophragma glabrum, L. parviflorum",Bee | Fly | Other or Unclassified,True,False,False,True,False,False,False
380,Rubus laciniatus,Bee | Butterfly,True,True,False,False,False,False,False
211,Impatiens glandulifera,Bee | Fly,True,False,False,True,False,False,False
9,Allium aflatuense,Bee,True,False,False,False,False,False,False
126,Diascia,Bee,True,False,False,False,False,False,False
70,Centaurea sp.,Bee | Fly | Wasp,True,False,False,True,False,True,False


In [156]:
pollinator_summary[
    pollinator_boolean_columns
].sum().sort_values(
    ascending=False
)

supports_bee_observed            447
supports_fly_observed            235
supports_wasp_observed           108
supports_butterfly_observed       53
supports_beetle_observed          39
supports_hummingbird_observed     14
supports_moth_observed             5
dtype: int64

In [157]:
print(
    "Duplicate plant keys:",
    pollinator_summary[
        "scientific_name_key"
    ].duplicated().sum(),
)

print(
    "Missing plant keys:",
    pollinator_summary[
        "scientific_name_key"
    ].isna().sum(),
)

print(
    "Total summarized observations:",
    pollinator_summary[
        "pollinator_observation_count"
    ].sum(),
)

print(
    "Original identified records:",
    len(pollinator_identified),
)

Duplicate plant keys: 0
Missing plant keys: 0
Total summarized observations: 34949
Original identified records: 34949


In [158]:
pollinator_identified.to_pickle(
    PROCESSED_DIR
    / "checkpoint_pollinator_clean.pkl"
)

pollinator_summary.to_pickle(
    PROCESSED_DIR
    / "checkpoint_pollinator_summary.pkl"
)

print("Pollinator checkpoints saved.")

Pollinator checkpoints saved.


In [159]:
FLY_FAMILIES.update({
    "Bombylidae",
    "Empididae",
    "Muscoidea",
    "Sarcophagidae",
    "Conopidae",
    "Scathophagidae",
    "Bibionidae",
    "Lonchaeidae",
    "Rhagionidae",
    "Nemestrinidae",
    "Asilidae",
    "Diptera",
})

BEETLE_FAMILIES.update({
    "Buprestidae",
    "Bruchidae",
    "Elateridae",
    "Cleridae",
    "Coleoptera",
})

MOTH_FAMILIES.update({
    "Adelidae",
    "Lepidoptera",
})

WASP_FAMILIES.update({
    "Cimbicidae",
    "Ichneumonidae",
    "Chrysididae",
    "Symphyta",
    "Cephidae",
    "Wasp",
})

In [160]:
BUTTERFLY_FAMILIES.add("Hesperidae")

In [161]:
pollinator_identified["pollinator_group"] = (
    pollinator_identified.apply(
        classify_pollinator_group,
        axis=1,
    )
)

pollinator_identified[
    "pollinator_group"
].value_counts()

pollinator_group
Bee                      27153
Fly                       5698
Beetle                     794
Wasp                       663
Other or Unclassified      228
Butterfly                  200
Moth                       194
Hummingbird                 19
Name: count, dtype: int64

In [162]:
def classify_plant_name_quality(name):
    if pd.isna(name):
        return "missing"

    text = str(name).strip()

    if "," in text or ";" in text or "/" in text:
        return "multiple_plants"

    words = text.split()

    if len(words) == 1:
        return "genus_only"

    if any(
        token.casefold() in {"sp.", "spp.", "unknown"}
        for token in words
    ):
        return "unspecified_species"

    if len(words) >= 2:
        return "species_or_infraspecific"

    return "review"

In [163]:
pollinator_identified["plant_name_quality"] = (
    pollinator_identified["plant_species"]
    .apply(classify_plant_name_quality)
)

pollinator_identified[
    "plant_name_quality"
].value_counts()

plant_name_quality
species_or_infraspecific    32279
genus_only                   2172
unspecified_species           268
multiple_plants               230
Name: count, dtype: int64

In [164]:
pollinator_mergeable = (
    pollinator_identified.loc[
        pollinator_identified[
            "plant_name_quality"
        ] == "species_or_infraspecific"
    ]
    .copy()
)

print(
    "Mergeable pollinator records:",
    len(pollinator_mergeable),
)

print(
    "Mergeable plant names:",
    pollinator_mergeable[
        "scientific_name_key"
    ].nunique(),
)

Mergeable pollinator records: 32279
Mergeable plant names: 325


In [165]:
pollinator_nonmergeable = (
    pollinator_identified.loc[
        pollinator_identified[
            "plant_name_quality"
        ] != "species_or_infraspecific"
    ]
    .copy()
)

pollinator_nonmergeable[
    [
        "plant_species",
        "plant_name_quality",
    ]
].drop_duplicates().to_csv(
    REVIEW_DIR
    / "nonmergeable_pollinator_plant_names.csv",
    index=False,
)

In [166]:
pollinator_summary = (
    pollinator_mergeable.groupby(
        "scientific_name_key",
        as_index=False,
    )
    .agg(aggregation_rules)
)

In [167]:
observation_counts = (
    pollinator_mergeable.groupby(
        "scientific_name_key"
    )[observation_id_column]
    .agg(
        pollinator_observation_count="size",
        unique_observation_count="nunique",
    )
    .reset_index()
)

In [168]:
pollinator_summary.columns.tolist()

['scientific_name_key',
 'plant_species',
 'pollinator_family',
 'pollinator_genus',
 'pollinator_species',
 'pollinator_group',
 'habitat',
 'location_name',
 'study']

In [169]:
observation_counts = (
    pollinator_mergeable.groupby(
        "scientific_name_key"
    )[observation_id_column]
    .agg(
        pollinator_observation_count="size",
        unique_observation_count="nunique",
    )
    .reset_index()
)

In [170]:
pollinator_summary = (
    pollinator_summary.merge(
        observation_counts,
        on="scientific_name_key",
        how="left",
        validate="one_to_one",
    )
)

In [171]:
pollinator_summary[
    [
        "scientific_name_key",
        "pollinator_observation_count",
        "unique_observation_count",
    ]
].head()

,scientific_name_key,pollinator_observation_count,unique_observation_count
0,achillea millefolium,748,748
1,achlys triphylla,2,2
2,agoseris aurantiaca,1,1
3,allium acuminatum,73,73
4,allium aflatuense,10,10


In [172]:
print(
    "Duplicate plant keys:",
    pollinator_summary[
        "scientific_name_key"
    ].duplicated().sum(),
)

print(
    "Summarized observations:",
    pollinator_summary[
        "pollinator_observation_count"
    ].sum(),
)

print(
    "Mergeable source records:",
    len(pollinator_mergeable),
)

Duplicate plant keys: 0
Summarized observations: 32279
Mergeable source records: 32279


In [173]:
for group in POLLINATOR_GROUPS:
    column_name = (
        "supports_"
        + group.casefold()
        + "_observed"
    )

    pollinator_summary[column_name] = (
        pollinator_summary[
            "pollinator_group"
        ]
        .apply(
            lambda value: contains_group(
                value,
                group,
            )
        )
    )

In [174]:
print("=" * 50)
print("DATASET SIZES")
print("=" * 50)

print(f"MoBot Plants        : {mobot['scientific_name_key'].nunique():,}")
print(f"Ladybird Plants     : {ladybird['preferred_scientific_name_key'].nunique():,}")
print(f"Pollinator Plants   : {pollinator_summary['scientific_name_key'].nunique():,}")

DATASET SIZES
MoBot Plants        : 997
Ladybird Plants     : 4,048
Pollinator Plants   : 325


In [175]:
mobot_set = set(
    mobot["scientific_name_key"].dropna().unique()
)

ladybird_set = set(
    ladybird["preferred_scientific_name_key"].dropna().unique()
)

pollinator_set = set(
    pollinator_summary["scientific_name_key"].dropna().unique()
)

In [176]:
print("=" * 50)
print("OVERLAP")
print("=" * 50)

print("MoBot ∩ Ladybird:", len(mobot_set & ladybird_set))
print("MoBot ∩ Pollinator:", len(mobot_set & pollinator_set))
print("Ladybird ∩ Pollinator:", len(ladybird_set & pollinator_set))

OVERLAP
MoBot ∩ Ladybird: 817
MoBot ∩ Pollinator: 22
Ladybird ∩ Pollinator: 116


In [177]:
mobot_ladybird = len(mobot_set & ladybird_set)

mobot_pollinator = len(mobot_set & pollinator_set)

print(
    f"Ladybird covers {mobot_ladybird / len(mobot_set) * 100:.1f}% "
    "of MoBot plants"
)

print(
    f"Pollinator dataset covers {mobot_pollinator / len(mobot_set) * 100:.1f}% "
    "of MoBot plants"
)

Ladybird covers 81.9% of MoBot plants
Pollinator dataset covers 2.2% of MoBot plants


In [178]:
missing_from_ladybird = sorted(
    mobot_set - ladybird_set
)

print(
    "Plants missing from Ladybird:",
    len(missing_from_ladybird)
)

missing_from_ladybird[:20]

Plants missing from Ladybird: 180


['acacia koa',
 'acalypha wilkesiana',
 'acer saccharum subsp. grandidentatum',
 'acer saccharum subsp. nigrum',
 'acoelorrhaphe wrightii',
 'acorus calamus',
 'aesculus parviflora var. serotina',
 'agave havardiana',
 'alnus incana subsp. rugosa',
 'amelanchier obovalis',
 'amorpha ouachitensis',
 'amsonia ciliata var. filifolia',
 'amsonia tabernaemontana var. salicifolia',
 'anemone americana',
 'antennaria dioica',
 'aquilegia caerulea',
 'aquilegia chrysantha var. hinckleyana',
 'aronia melanocarpa var. elata',
 'artemisia vulgaris',
 'asarum shuttleworthii']

In [179]:
missing_from_pollinator = sorted(
    mobot_set - pollinator_set
)

print(
    "Plants missing from Pollinator:",
    len(missing_from_pollinator)
)

missing_from_pollinator[:20]

Plants missing from Pollinator: 975


['abies balsamea',
 'abies concolor',
 'abies fraseri',
 'abies grandis',
 'acacia koa',
 'acalypha wilkesiana',
 'acer macrophyllum',
 'acer negundo',
 'acer pensylvanicum',
 'acer rubrum',
 'acer saccharinum',
 'acer saccharum',
 'acer saccharum subsp. grandidentatum',
 'acer saccharum subsp. nigrum',
 'acoelorrhaphe wrightii',
 'acorus calamus',
 'actaea pachypoda',
 'actaea racemosa',
 'adiantum capillus-veneris',
 'adiantum pedatum']

In [180]:
missing_ladybird_df = pd.DataFrame({
    "scientific_name_key": missing_from_ladybird
})

missing_pollinator_df = pd.DataFrame({
    "scientific_name_key": missing_from_pollinator
})

display(missing_ladybird_df.head())

display(missing_pollinator_df.head())

,scientific_name_key
0,acacia koa
1,acalypha wilkesiana
2,acer saccharum subsp. grandidentatum
3,acer saccharum subsp. nigrum
4,acoelorrhaphe wrightii


,scientific_name_key
0,abies balsamea
1,abies concolor
2,abies fraseri
3,abies grandis
4,acacia koa


In [181]:
missing_ladybird_df.to_csv(
    REVIEW_DIR / "missing_from_ladybird.csv",
    index=False
)

missing_pollinator_df.to_csv(
    REVIEW_DIR / "missing_from_pollinator.csv",
    index=False
)

print("Coverage review files saved.")

Coverage review files saved.


In [182]:
complete_overlap = (
    mobot_set
    & ladybird_set
    & pollinator_set
)

print(
    "Plants found in all datasets:",
    len(complete_overlap)
)

Plants found in all datasets: 22


In [183]:
coverage_report = pd.DataFrame({
    "Dataset": [
        "MoBot",
        "Ladybird",
        "Pollinator"
    ],
    "Unique Plants": [
        len(mobot_set),
        len(ladybird_set),
        len(pollinator_set)
    ]
})

coverage_report

,Dataset,Unique Plants
0,MoBot,997
1,Ladybird,4048
2,Pollinator,325


In [184]:
def combine_unique(series):
    """
    Combine unique non-null values into a pipe-separated string.
    """
    values = sorted(
        {
            str(v).strip()
            for v in series.dropna()
            if str(v).strip()
        }
    )

    if not values:
        return pd.NA

    return " | ".join(values)

In [185]:
mobot_master = (
    mobot.groupby(
        "scientific_name_key",
        as_index=False
    )
    .agg({
        "plant_id": "first",
        "scientific_name": "first",
        "common_name": combine_unique,
        "plant_type": combine_unique,
        "sunlight": combine_unique,
        "moisture": combine_unique,
        "maintenance": combine_unique,
        "zone_min": "min",
        "zone_max": "max",
        "bloom_start": combine_unique,
        "bloom_end": combine_unique,
        "native_state": combine_unique
    })
)

print(mobot_master.shape)

mobot_master.head()

(997, 13)


,scientific_name_key,plant_id,scientific_name,common_name,plant_type,sunlight,moisture,maintenance,zone_min,zone_max,bloom_start,bloom_end,native_state
0,abies balsamea,PLANT-00001,Abies balsamea,balsam fir,Needled evergreen,Full sun to part shade,Medium,Medium,3.0,6.0,Non-flowering,Non-flowering,Connecticut | Iowa | Maine | Massachusetts | Michigan | Minnesota | New Hampshire | New Jersey | New York | Pennsylvania | Vermont | Virginia | We...
1,abies concolor,PLANT-00002,Abies concolor,white fir,Needled evergreen,Full sun to part shade,Medium,Medium,3.0,7.0,Non-flowering,Non-flowering,Arizona | California | Colorado | Idaho | Nevada | Oregon | Utah | Wyoming
2,abies fraseri,PLANT-00003,Abies fraseri,Fraser fir,Needled evergreen,Full sun to part shade,Medium,Medium,4.0,7.0,Non-flowering,Non-flowering,Georgia | Minnesota | North Carolina | Tennessee | Virginia
3,abies grandis,PLANT-00004,Abies grandis,grand fir,Needled evergreen,Full sun to part shade,Medium,Medium,5.0,6.0,Non-flowering,Non-flowering,California | Idaho | Montana | Oregon | Washington
4,acacia koa,PLANT-00005,Acacia koa,koa,Tree,Full sun,Medium,Medium,10.0,11.0,Seasonal bloomer,Seasonal bloomer,Hawaii


In [186]:
mobot_master = mobot_master.rename(
    columns={
        "native_state": "native_states"
    }
)

In [188]:
duplicate_preferred_keys = (
    ladybird.loc[
        ladybird["preferred_scientific_name_key"].duplicated(
            keep=False
        )
    ]
    .sort_values("preferred_scientific_name_key")
)

duplicate_preferred_keys[
    [
        "scientific_name_ladybird",
        "scientific_name_key_ladybird",
        "usda_code",
        "accepted_usda_code",
        "code_type",
        "scientific_name_usda",
        "preferred_scientific_name_key",
    ]
]

,scientific_name_ladybird,scientific_name_key_ladybird,usda_code,accepted_usda_code,code_type,scientific_name_usda,preferred_scientific_name_key
307,Celtis tenuifolia,celtis tenuifolia,CETE,CEPU10,synonym,Celtis pumila,celtis pumila
3757,Celtis pumila,celtis pumila,CEPU10,CEPU10,accepted,Celtis pumila,celtis pumila


In [189]:
master = mobot_master.merge(
    ladybird,
    left_on="scientific_name_key",
    right_on="scientific_name_key_ladybird",
    how="left",
    validate="one_to_one",
    indicator="ladybird_direct_merge",
    suffixes=("_mobot", "_ladybird"),
)

print("Rows after direct merge:", len(master))

master["ladybird_direct_merge"].value_counts()

Rows after direct merge: 997


ladybird_direct_merge
both          817
left_only     180
right_only      0
Name: count, dtype: int64

In [190]:
unmatched_mask = (
    master["ladybird_direct_merge"] == "left_only"
)

print(
    "Unmatched after direct scientific-name merge:",
    unmatched_mask.sum(),
)

Unmatched after direct scientific-name merge: 180


In [191]:
ladybird_fallback = ladybird.copy()

ladybird_fallback["name_matches_accepted"] = (
    ladybird_fallback["scientific_name_key_ladybird"]
    == ladybird_fallback["preferred_scientific_name_key"]
)

ladybird_fallback["accepted_code_priority"] = (
    ladybird_fallback["code_type"]
    .map({
        "accepted": 1,
        "synonym": 2,
    })
    .fillna(3)
)

ladybird_fallback["record_completeness"] = (
    ladybird_fallback.notna().sum(axis=1)
)

ladybird_fallback = (
    ladybird_fallback
    .sort_values(
        [
            "preferred_scientific_name_key",
            "name_matches_accepted",
            "accepted_code_priority",
            "record_completeness",
        ],
        ascending=[
            True,
            False,
            True,
            False,
        ],
    )
    .drop_duplicates(
        subset="preferred_scientific_name_key",
        keep="first",
    )
    .drop(
        columns=[
            "name_matches_accepted",
            "accepted_code_priority",
            "record_completeness",
        ]
    )
)

print(
    "Duplicate fallback keys:",
    ladybird_fallback[
        "preferred_scientific_name_key"
    ].duplicated().sum(),
)

Duplicate fallback keys: 0


In [192]:
fallback_matches = (
    mobot_master.loc[
        mobot_master["scientific_name_key"].isin(
            master.loc[
                unmatched_mask,
                "scientific_name_key",
            ]
        )
    ]
    .merge(
        ladybird_fallback,
        left_on="scientific_name_key",
        right_on="preferred_scientific_name_key",
        how="left",
        validate="one_to_one",
        indicator="ladybird_fallback_merge",
    )
)

fallback_matches[
    "ladybird_fallback_merge"
].value_counts()

ladybird_fallback_merge
left_only     180
right_only      0
both            0
Name: count, dtype: int64

In [193]:
master = mobot_master.merge(
    ladybird,
    left_on="scientific_name_key",
    right_on="scientific_name_key_ladybird",
    how="left",
    validate="one_to_one",
    indicator="ladybird_merge_status",
    suffixes=("_mobot", "_ladybird"),
)

In [194]:
print("Master rows:", len(master))

print(
    master["ladybird_merge_status"]
    .value_counts()
)

print(
    "Duplicate MoBot plant keys:",
    master[
        "scientific_name_key"
    ].duplicated().sum(),
)

Master rows: 997
ladybird_merge_status
both          817
left_only     180
right_only      0
Name: count, dtype: int64
Duplicate MoBot plant keys: 0


In [195]:
master = master.merge(
    pollinator_summary,
    on="scientific_name_key",
    how="left",
    validate="one_to_one",
    suffixes=("", "_pollinator"),
    indicator="pollinator_merge_status",
)

print("Master rows:", len(master))
print(master["pollinator_merge_status"].value_counts())

print(
    "Duplicate plant keys:",
    master["scientific_name_key"].duplicated().sum(),
)

Master rows: 997
pollinator_merge_status
left_only     975
both           22
right_only      0
Name: count, dtype: int64
Duplicate plant keys: 0


In [196]:
plant_states = (
    mobot[
        [
            "plant_id",
            "scientific_name_key",
            "native_state",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "plant_id",
            "native_state",
        ]
    )
    .reset_index(drop=True)
)

print("Plant-state rows:", len(plant_states))
print(
    "Duplicate plant-state pairs:",
    plant_states.duplicated(
        subset=["plant_id", "native_state"]
    ).sum(),
)

plant_states.head()

Plant-state rows: 20091
Duplicate plant-state pairs: 0


,plant_id,scientific_name_key,native_state
0,PLANT-00001,abies balsamea,Connecticut
1,PLANT-00001,abies balsamea,Iowa
2,PLANT-00001,abies balsamea,Maine
3,PLANT-00001,abies balsamea,Massachusetts
4,PLANT-00001,abies balsamea,Michigan


In [197]:
master["has_ladybird_data"] = (
    master["ladybird_merge_status"] == "both"
)

master["has_pollinator_data"] = (
    master["pollinator_merge_status"] == "both"
)

In [198]:
pollinator_boolean_columns = [
    column
    for column in master.columns
    if column.startswith("supports_")
    and column.endswith("_observed")
]

for column in pollinator_boolean_columns:
    master[column] = (
        master[column]
        .fillna(False)
        .astype(bool)
    )

In [199]:
print("Rows:", len(master))
print(
    "Unique plants:",
    master["scientific_name_key"].nunique(),
)
print(
    "Unique IDs:",
    master["plant_id"].nunique(),
)
print(
    "Ladybird coverage:",
    master["has_ladybird_data"].sum(),
)
print(
    "Pollinator coverage:",
    master["has_pollinator_data"].sum(),
)


Rows: 997
Unique plants: 997
Unique IDs: 997
Ladybird coverage: 817
Pollinator coverage: 22


In [200]:
quality_report = pd.DataFrame({
    "metric": [
        "Total native plants",
        "Plants with Ladybird data",
        "Plants without Ladybird data",
        "Plants with pollinator data",
        "Plants without pollinator data",
        "Unique plant IDs",
        "Duplicate plant IDs",
    ],
    "value": [
        len(master),
        master["has_ladybird_data"].sum(),
        (~master["has_ladybird_data"]).sum(),
        master["has_pollinator_data"].sum(),
        (~master["has_pollinator_data"]).sum(),
        master["plant_id"].nunique(),
        master["plant_id"].duplicated().sum(),
    ],
})

quality_report

,metric,value
0,Total native plants,997
1,Plants with Ladybird data,817
2,Plants without Ladybird data,180
3,Plants with pollinator data,22
4,Plants without pollinator data,975
5,Unique plant IDs,997
6,Duplicate plant IDs,0


In [201]:
final_columns = [
    "plant_id",
    "scientific_name_key",
    "scientific_name_mobot",
    "common_name_mobot",
    "native_states",
    "plant_type",
    "sunlight",
    "moisture",
    "maintenance",
    "zone_min",
    "zone_max",
    "bloom_start",
    "bloom_end",

    "accepted_usda_code",
    "plant_family",
    "preferred_scientific_name",
    "preferred_common_name",

    "growth_habit",
    "duration",
    "size_notes",
    "bloom_color",
    "bloom_time",
    "water_use",
    "light_requirements",
    "soil_moisture",
    "soil_ph",
    "soil_description",
    "wildlife_use",
    "commercial_availability",
    "propagation_description",

    "supports_full_sun",
    "supports_part_shade",
    "supports_shade",
    "supports_low_water",
    "supports_medium_water",
    "supports_high_water",

    "bloom_jan",
    "bloom_feb",
    "bloom_mar",
    "bloom_apr",
    "bloom_may",
    "bloom_jun",
    "bloom_jul",
    "bloom_aug",
    "bloom_sep",
    "bloom_oct",
    "bloom_nov",
    "bloom_dec",

    "pollinator_group",
    "pollinator_observation_count",
    "supports_bee_observed",
    "supports_butterfly_observed",
    "supports_moth_observed",
    "supports_fly_observed",
    "supports_beetle_observed",
    "supports_wasp_observed",
    "supports_hummingbird_observed",

    "has_ladybird_data",
    "has_pollinator_data",
]

In [202]:
final_columns = [
    column
    for column in final_columns
    if column in master.columns
]

plants_master = master[final_columns].copy()

In [204]:
def first_existing_series(df, candidates):
    for column in candidates:
        if column in df.columns:
            return df[column]
    return pd.Series(pd.NA, index=df.index)


plants_master["scientific_name"] = (
    first_existing_series(
        plants_master,
        [
            "preferred_scientific_name",
            "scientific_name_ladybird",
            "scientific_name_mobot",
            "scientific_name",
        ],
    )
)

plants_master["common_name"] = (
    first_existing_series(
        plants_master,
        [
            "preferred_common_name",
            "common_name_ladybird",
            "common_name_mobot",
            "common_name",
        ],
    )
)

In [205]:
def combine_available_columns(df, candidates):
    available = [
        column
        for column in candidates
        if column in df.columns
    ]

    if not available:
        return pd.Series(pd.NA, index=df.index)

    result = df[available[0]].copy()

    for column in available[1:]:
        result = result.combine_first(df[column])

    return result


plants_master["scientific_name"] = (
    combine_available_columns(
        plants_master,
        [
            "preferred_scientific_name",
            "scientific_name_ladybird",
            "scientific_name_mobot",
            "scientific_name",
        ],
    )
)

plants_master["common_name"] = (
    combine_available_columns(
        plants_master,
        [
            "preferred_common_name",
            "common_name_ladybird",
            "common_name_mobot",
            "common_name",
        ],
    )
)

In [206]:
print(
    "Missing scientific names:",
    plants_master["scientific_name"].isna().sum(),
)

print(
    "Missing common names:",
    plants_master["common_name"].isna().sum(),
)

Missing scientific names: 180
Missing common names: 180


In [207]:
[
    column
    for column in master.columns
    if "scientific_name" in column
    or "common_name" in column
]

['scientific_name_key',
 'scientific_name',
 'common_name',
 'scientific_name_ladybird',
 'common_name_ladybird',
 'scientific_name_key_ladybird',
 'scientific_name_with_author',
 'scientific_name_usda',
 'scientific_name_key_usda',
 'common_name_usda',
 'preferred_scientific_name',
 'preferred_scientific_name_key',
 'preferred_common_name']

In [208]:
mobot_master.columns.tolist()

['scientific_name_key',
 'plant_id',
 'scientific_name',
 'common_name',
 'plant_type',
 'sunlight',
 'moisture',
 'maintenance',
 'zone_min',
 'zone_max',
 'bloom_start',
 'bloom_end',
 'native_states']

In [209]:
def combine_available_columns(df, candidates):
    available = [
        column
        for column in candidates
        if column in df.columns
    ]

    print("Using columns:", available)

    if not available:
        return pd.Series(pd.NA, index=df.index)

    result = df[available[0]].copy()

    for column in available[1:]:
        result = result.combine_first(df[column])

    return result

In [210]:
master["final_scientific_name"] = (
    combine_available_columns(
        master,
        [
            "preferred_scientific_name",
            "scientific_name_ladybird",
            "scientific_name_usda",
            "scientific_name_mobot",
            "scientific_name_x",
            "scientific_name",
        ],
    )
)

Using columns: ['preferred_scientific_name', 'scientific_name_ladybird', 'scientific_name_usda', 'scientific_name']


In [211]:
master["final_common_name"] = (
    combine_available_columns(
        master,
        [
            "preferred_common_name",
            "common_name_ladybird",
            "common_name_usda",
            "common_name_mobot",
            "common_name_x",
            "common_name",
        ],
    )
)

Using columns: ['preferred_common_name', 'common_name_ladybird', 'common_name_usda', 'common_name']


In [212]:
print(
    "Missing final scientific names:",
    master["final_scientific_name"].isna().sum(),
)

print(
    "Missing final common names:",
    master["final_common_name"].isna().sum(),
)

Missing final scientific names: 0
Missing final common names: 1


In [213]:
master.loc[
    ~master["has_ladybird_data"],
    [
        "scientific_name_key",
        "final_scientific_name",
        "final_common_name",
    ],
].head(20)

,scientific_name_key,final_scientific_name,final_common_name
4,acacia koa,Acacia koa,koa
5,acalypha wilkesiana,Acalypha wilkesiana,Jacob's coat
12,acer saccharum subsp. grandidentatum,Acer saccharum subsp. grandidentatum,bigtooth maple
13,acer saccharum subsp. nigrum,Acer saccharum subsp. nigrum,black maple
15,acoelorrhaphe wrightii,Acoelorrhaphe wrightii,paurotis palm
16,acorus calamus,Acorus calamus,sweet flag
25,aesculus parviflora var. serotina,Aesculus parviflora var. serotina,bottlebrush buckeye
34,agave havardiana,Agave havardiana,century plant
43,alnus incana subsp. rugosa,Alnus incana subsp. rugosa,hazel alder
50,amelanchier obovalis,Amelanchier obovalis,coastal serviceberry


In [214]:
final_columns = [
    "plant_id",
    "scientific_name_key",
    "final_scientific_name",
    "final_common_name",
    "native_states",
    "plant_type",
    "sunlight",
    "moisture",
    "maintenance",
    "zone_min",
    "zone_max",
    "bloom_start",
    "bloom_end",

    "accepted_usda_code",
    "plant_family",
    "growth_habit",
    "duration",
    "size_notes",
    "bloom_color",
    "bloom_time",
    "water_use",
    "light_requirements",
    "soil_moisture",
    "soil_ph",
    "soil_description",
    "wildlife_use",
    "commercial_availability",
    "propagation_description",

    "supports_full_sun",
    "supports_part_shade",
    "supports_shade",
    "supports_low_water",
    "supports_medium_water",
    "supports_high_water",

    "bloom_jan",
    "bloom_feb",
    "bloom_mar",
    "bloom_apr",
    "bloom_may",
    "bloom_jun",
    "bloom_jul",
    "bloom_aug",
    "bloom_sep",
    "bloom_oct",
    "bloom_nov",
    "bloom_dec",

    "pollinator_group",
    "pollinator_observation_count",
    "supports_bee_observed",
    "supports_butterfly_observed",
    "supports_moth_observed",
    "supports_fly_observed",
    "supports_beetle_observed",
    "supports_wasp_observed",
    "supports_hummingbird_observed",

    "has_ladybird_data",
    "has_pollinator_data",
]

In [215]:
final_columns = [
    column
    for column in final_columns
    if column in master.columns
]

plants_master = master[final_columns].copy()

In [216]:
plants_master = plants_master.rename(
    columns={
        "final_scientific_name": "scientific_name",
        "final_common_name": "common_name",
    }
)

In [217]:
print("Rows:", len(plants_master))

print(
    "Missing scientific names:",
    plants_master["scientific_name"].isna().sum(),
)

print(
    "Missing common names:",
    plants_master["common_name"].isna().sum(),
)

print(
    "Duplicate plant IDs:",
    plants_master["plant_id"].duplicated().sum(),
)

Rows: 997
Missing scientific names: 0
Missing common names: 1
Duplicate plant IDs: 0


In [218]:
plants_master.loc[
    plants_master["common_name"].isna(),
    [
        "plant_id",
        "scientific_name",
        "scientific_name_key",
        "plant_family",
    ],
]

,plant_id,scientific_name,scientific_name_key,plant_family
568,PLANT-00569,Matteuccia struthiopteris subsp. pensylvanica,matteuccia struthiopteris subsp. pensylvanica,NaN


In [219]:
plants_master["display_name"] = (
    plants_master["common_name"]
    .combine_first(
        plants_master["scientific_name"]
    )
)

In [220]:
plants_master.to_csv(
    PROCESSED_DIR / "plants_master.csv",
    index=False,
)

plant_states.to_csv(
    PROCESSED_DIR / "plant_states.csv",
    index=False,
)

quality_report.to_csv(
    PROCESSED_DIR / "data_quality_report.csv",
    index=False,
)

print("Final datasets exported successfully.")

Final datasets exported successfully.


In [221]:
print("Master file:", PROCESSED_DIR / "plants_master.csv")
print("Plant-state file:", PROCESSED_DIR / "plant_states.csv")
print("Quality report:", PROCESSED_DIR / "data_quality_report.csv")

Master file: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\processed\plants_master.csv
Plant-state file: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\processed\plant_states.csv
Quality report: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\processed\data_quality_report.csv


In [222]:
plants_master.head(10)

,plant_id,scientific_name_key,scientific_name,common_name,native_states,plant_type,sunlight,moisture,maintenance,zone_min,zone_max,bloom_start,bloom_end,accepted_usda_code,plant_family,growth_habit,duration,size_notes,bloom_color,bloom_time,water_use,light_requirements,soil_moisture,soil_ph,soil_description,wildlife_use,commercial_availability,propagation_description,supports_full_sun,supports_part_shade,supports_shade,supports_low_water,supports_medium_water,supports_high_water,bloom_jan,bloom_feb,bloom_mar,bloom_apr,bloom_may,bloom_jun,bloom_jul,bloom_aug,bloom_sep,bloom_oct,bloom_nov,bloom_dec,pollinator_group,pollinator_observation_count,supports_bee_observed,supports_butterfly_observed,supports_moth_observed,supports_fly_observed,supports_beetle_observed,supports_wasp_observed,supports_hummingbird_observed,has_ladybird_data,has_pollinator_data,display_name
0,PLANT-00001,abies balsamea,Abies balsamea,"Balsam Fir, Blister Pine, Northern Balsam",Connecticut | Iowa | Maine | Massachusetts | Michigan | Minnesota | New Hampshire | New Jersey | New York | Pennsylvania | Vermont | Virginia | We...,Needled evergreen,Full sun to part shade,Medium,Medium,3.0,6.0,Non-flowering,Non-flowering,ABBA,Pinaceae,Tree,Perennial,Up to about 75 feet tall.,"Yellow , Green , Purple , Brown","Sep , Oct , Nov",Medium,"Sun , Part Shade , Shade",Moist,Acidic (pH<6.8),"Well-drained, acid, moist soils.",Songbirds and squirrels eat seed and deer browse foliage. Deer and moose browse the foliage in winter.,yes,"Abies spp. are best propagated by means of seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields.",True,True,True,False,True,False,False,False,False,False,False,False,False,False,True,True,True,False,NaN,NaN,False,False,False,False,False,False,False,True,False,"Balsam Fir, Blister Pine, Northern Balsam"
1,PLANT-00002,abies concolor,Abies concolor,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",Arizona | California | Colorado | Idaho | Nevada | Oregon | Utah | Wyoming,Needled evergreen,Full sun to part shade,Medium,Medium,3.0,7.0,Non-flowering,Non-flowering,ABCO,Pinaceae,Tree,Perennial,"Up to about 130 feet tall, spread up to about 60 feet.",Red,"Apr , May , Jun",Medium,"Sun , Part Shade",<NA>,<NA>,"Well-drained, gravelly or sandy-loam soils.","The winged seeds of this and other firs are eaten by songbirds and various mammals, especially squirrels and chipmunks. Deer and grouse feed on th...",yes,"Seed is the easiest method of propagation. In nature, Abies seeds often germinate on melting snow fields. Cuttings should be taken from December t...",True,True,False,False,True,False,False,False,False,True,True,True,False,False,False,False,False,False,NaN,NaN,False,False,False,False,False,False,False,True,False,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California"
2,PLANT-00003,abies fraseri,Abies fraseri,"Fraser Fir, She-balsam",Georgia | Minnesota | North Carolina | Tennessee | Virginia,Needled evergreen,Full sun to part shade,Medium,Medium,4.0,7.0,Non-flowering,Non-flowering,ABFR,Pinaceae,Tree,Perennial,Up to about 75 feet tall.,Purple,Apr,Medium,Shade,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,NaN,NaN,False,False,False,False,False,False,False,True,False,"Fraser Fir, She-balsam"
3,PLANT-00004,abies grandis,Abies grandis,"Grand Fir, Giant Fir",California | Idaho | Montana | Oregon | Washington,Needled evergreen,Full sun to part shade,Medium,Medium,5.0,6.0,Non-flowering,Non-flowering,ABGR,Pinaceae,Tree,<NA>,Up to more than 200 feet tall.,"White , Green","Apr , May",Medium,"Part Shade , Shade","Dry , Moist",<NA>,Well-drained soils.,<NA>,yes,"Abies spp. are best propagated by seeds sown in early spring. In nature, Abies seeds often germinate on melting snow fields.",False,True,True,False,True,False,False,False,False,True,True,False,F

In [223]:
identity_columns = [
    "plant_id",
    "scientific_name",
    "common_name",
    "display_name",
    "plant_family",
]

In [224]:
location_columns = [
    "native_states",
]

In [225]:
growing_columns = [
    "growth_habit",
    "plant_type",
    "zone_min",
    "zone_max",
    "maintenance",
    "soil_description",
    "soil_moisture",
    "soil_ph",
]

In [226]:
sunlight_columns = [
    "sunlight",
    "light_requirements",

    "supports_full_sun",
    "supports_part_shade",
    "supports_shade",
]

In [227]:
water_columns = [
    "moisture",
    "water_use",

    "supports_low_water",
    "supports_medium_water",
    "supports_high_water",
]

In [228]:
bloom_columns = [
    "bloom_start",
    "bloom_end",
    "bloom_color",
    "bloom_time",

    "bloom_jan",
    "bloom_feb",
    "bloom_mar",
    "bloom_apr",
    "bloom_may",
    "bloom_jun",
    "bloom_jul",
    "bloom_aug",
    "bloom_sep",
    "bloom_oct",
    "bloom_nov",
    "bloom_dec",
]

In [229]:
pollinator_columns = [
    "pollinator_group",

    "supports_bee_observed",
    "supports_butterfly_observed",
    "supports_moth_observed",
    "supports_fly_observed",
    "supports_beetle_observed",
    "supports_wasp_observed",
    "supports_hummingbird_observed",
]

In [230]:
status_columns = [
    "has_ladybird_data",
    "has_pollinator_data",
]

In [231]:
app_columns = (
    identity_columns
    + location_columns
    + growing_columns
    + sunlight_columns
    + water_columns
    + bloom_columns
    + pollinator_columns
    + status_columns
)

# Keep only columns that actually exist
app_columns = [
    column
    for column in app_columns
    if column in plants_master.columns
]

plants_app = plants_master[app_columns].copy()

print("Rows:", len(plants_app))
print("Columns:", len(plants_app.columns))

plants_app.head()

Rows: 997
Columns: 50


,plant_id,scientific_name,common_name,display_name,plant_family,native_states,growth_habit,plant_type,zone_min,zone_max,maintenance,soil_description,soil_moisture,soil_ph,sunlight,light_requirements,supports_full_sun,supports_part_shade,supports_shade,moisture,water_use,supports_low_water,supports_medium_water,supports_high_water,bloom_start,bloom_end,bloom_color,bloom_time,bloom_jan,bloom_feb,bloom_mar,bloom_apr,bloom_may,bloom_jun,bloom_jul,bloom_aug,bloom_sep,bloom_oct,bloom_nov,bloom_dec,pollinator_group,supports_bee_observed,supports_butterfly_observed,supports_moth_observed,supports_fly_observed,supports_beetle_observed,supports_wasp_observed,supports_hummingbird_observed,has_ladybird_data,has_pollinator_data
0,PLANT-00001,Abies balsamea,"Balsam Fir, Blister Pine, Northern Balsam","Balsam Fir, Blister Pine, Northern Balsam",Pinaceae,Connecticut | Iowa | Maine | Massachusetts | Michigan | Minnesota | New Hampshire | New Jersey | New York | Pennsylvania | Vermont | Virginia | We...,Tree,Needled evergreen,3.0,6.0,Medium,"Well-drained, acid, moist soils.",Moist,Acidic (pH<6.8),Full sun to part shade,"Sun , Part Shade , Shade",True,True,True,Medium,Medium,False,True,False,Non-flowering,Non-flowering,"Yellow , Green , Purple , Brown","Sep , Oct , Nov",False,False,False,False,False,False,False,False,True,True,True,False,NaN,False,False,False,False,False,False,False,True,False
1,PLANT-00002,Abies concolor,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California","White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",Pinaceae,Arizona | California | Colorado | Idaho | Nevada | Oregon | Utah | Wyoming,Tree,Needled evergreen,3.0,7.0,Medium,"Well-drained, gravelly or sandy-loam soils.",<NA>,<NA>,Full sun to part shade,"Sun , Part Shade",True,True,False,Medium,Medium,False,True,False,Non-flowering,Non-flowering,Red,"Apr , May , Jun",False,False,False,True,True,True,False,False,False,False,False,False,NaN,False,False,False,False,False,False,False,True,False
2,PLANT-00003,Abies fraseri,"Fraser Fir, She-balsam","Fraser Fir, She-balsam",Pinaceae,Georgia | Minnesota | North Carolina | Tennessee | Virginia,Tree,Needled evergreen,4.0,7.0,Medium,<NA>,<NA>,<NA>,Full sun to part shade,Shade,False,False,True,Medium,Medium,False,True,False,Non-flowering,Non-flowering,Purple,Apr,False,False,False,True,False,False,False,False,False,False,False,False,NaN,False,False,False,False,False,False,False,True,False
3,PLANT-00004,Abies grandis,"Grand Fir, Giant Fir","Grand Fir, Giant Fir",Pinaceae,California | Idaho | Montana | Oregon | Washington,Tree,Needled evergreen,5.0,6.0,Medium,Well-drained soils.,"Dry , Moist",<NA>,Full sun to part shade,"Part Shade , Shade",False,True,True,Medium,Medium,False,True,False,Non-flowering,Non-flowering,"White , Green","Apr , May",False,False,False,True,True,False,False,False,False,False,False,False,NaN,False,False,False,False,False,False,False,True,False
4,PLANT-00005,Acacia koa,koa,koa,NaN,Hawaii,NaN,Tree,10.0,11.0,Medium,NaN,NaN,NaN,Full sun,NaN,NaN,NaN,NaN,Medium,NaN,NaN,NaN,NaN,Seasonal bloomer,Seasonal bloomer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,False,False


In [232]:
plants_app.to_csv(
    PROCESSED_DIR / "plants_app.csv",
    index=False
)

print("Application dataset saved.")

Application dataset saved.


In [233]:
plants_app["image_url"] = pd.NA
plants_app["image_source"] = pd.NA

In [234]:
preferred_order = [

    # ---------- Identity ----------
    "plant_id",
    "scientific_name",
    "common_name",
    "display_name",
    "plant_family",

    # ---------- Images ----------
    "image_url",
    "image_source",

    # ---------- Geography ----------
    "native_states",

    # ---------- Growing ----------
    "growth_habit",
    "plant_type",

    "zone_min",
    "zone_max",

    "maintenance",

    "soil_description",
    "soil_moisture",
    "soil_ph",

    # ---------- Sun ----------
    "supports_full_sun",
    "supports_part_shade",
    "supports_shade",

    # ---------- Water ----------
    "supports_low_water",
    "supports_medium_water",
    "supports_high_water",

    # ---------- Bloom ----------
    "bloom_color",

    "bloom_jan",
    "bloom_feb",
    "bloom_mar",
    "bloom_apr",
    "bloom_may",
    "bloom_jun",
    "bloom_jul",
    "bloom_aug",
    "bloom_sep",
    "bloom_oct",
    "bloom_nov",
    "bloom_dec",

    # ---------- Pollinators ----------
    "supports_bee_observed",
    "supports_butterfly_observed",
    "supports_hummingbird_observed",
    "supports_moth_observed",
    "supports_fly_observed",
    "supports_beetle_observed",
    "supports_wasp_observed",

    # ---------- Extra ----------
    "has_ladybird_data",
    "has_pollinator_data"
]

In [235]:
preferred_order = [
    c for c in preferred_order
    if c in plants_app.columns
]

plants_app = plants_app[preferred_order]

In [236]:
plants_app.to_csv(
    PROCESSED_DIR / "plants_app.csv",
    index=False
)

print("Application dataset saved.")

Application dataset saved.


In [238]:
test2 = plants_app[
    plants_app["supports_butterfly_observed"]
]

print(len(test2))

test2[
    [
        "common_name",
        "scientific_name"
    ]
].head(15)

3


,common_name,scientific_name
14,"Common Yarrow, Western Yarrow, Yarrow, Milfoil",Achillea millefolium
83,"Kinnikinnick, Red Bearberry, Kinnikinnik",Arctostaphylos uva-ursi
178,"Large Camas, Leichtlin's Camas",Camassia leichtlinii


In [239]:
test3 = plants_app[
    plants_app["supports_bee_observed"]
]

print(len(test3))

test3[
    [
        "common_name",
        "scientific_name"
    ]
].head(15)

22


,common_name,scientific_name
14,"Common Yarrow, Western Yarrow, Yarrow, Milfoil",Achillea millefolium
39,Nodding Onion,Allium cernuum
40,Wild Chives,Allium schoenoprasum
61,"Western Pearly Everlasting, Pearly-everlasting",Anaphalis margaritacea
82,"Pacific Madrone, Oregon Laurel, Laurelwood",Arbutus menziesii
83,"Kinnikinnick, Red Bearberry, Kinnikinnik",Arctostaphylos uva-ursi
178,"Large Camas, Leichtlin's Camas",Camassia leichtlinii
182,"Bluebell Bellflower, Bluebell Of Scotland, Bluebell, Harebell, Witches' Thimble",Campanula rotundifolia
247,"Fireweed, Willow Herb, Great Willow Herb",Chamerion angustifolium
378,"California Poppy, California Gold Poppy",Eschscholzia californica


In [241]:
test5 = plants_app[
    plants_app["native_states"].str.contains(
        "Colorado",
        na=False
    )
]

print(len(test5))

test5[
    [
        "common_name",
        "scientific_name"
    ]
].head(20)

184


,common_name,scientific_name
1,"White Fir, Balsam Fir, Colorado Fir, Concolor Fir, Silver Fir, White Balsam, Oyamel De California",Abies concolor
7,"Box Elder, Box Elder Maple, Ash-leaved Maple, Ashleaf Maple, Red River Maple, Fresno De Guajuco",Acer negundo
12,bigtooth maple,Acer saccharum subsp. grandidentatum
14,"Common Yarrow, Western Yarrow, Yarrow, Milfoil",Achillea millefolium
19,"Southern Maidenhair Fern, Common Maidenhair Fern, Maidenhair Fern, Venus Hair Fern",Adiantum capillus-veneris
28,Slenderleaf False Foxglove,Agalinis tenuifolia
30,"Blue Giant Hyssop, Blue Giant-hyssop, Fragrant Giant Hyssop, Lavender Hyssop, Anise Hyssop",Agastache foeniculum
37,White Snakeroot,Ageratina altissima
38,Northern Water Plantain,Alisma triviale
39,Nodding Onion,Allium cernuum


In [242]:
recommendation_boolean_columns = [
    # Sunlight
    "supports_full_sun",
    "supports_part_shade",
    "supports_shade",

    # Water
    "supports_low_water",
    "supports_medium_water",
    "supports_high_water",

    # Bloom months
    "bloom_jan",
    "bloom_feb",
    "bloom_mar",
    "bloom_apr",
    "bloom_may",
    "bloom_jun",
    "bloom_jul",
    "bloom_aug",
    "bloom_sep",
    "bloom_oct",
    "bloom_nov",
    "bloom_dec",

    # Pollinators
    "supports_bee_observed",
    "supports_butterfly_observed",
    "supports_moth_observed",
    "supports_fly_observed",
    "supports_beetle_observed",
    "supports_wasp_observed",
    "supports_hummingbird_observed",

    # Data availability
    "has_ladybird_data",
    "has_pollinator_data",
]

recommendation_boolean_columns = [
    column
    for column in recommendation_boolean_columns
    if column in plants_app.columns
]

for column in recommendation_boolean_columns:
    plants_app[column] = (
        plants_app[column]
        .fillna(False)
        .astype(bool)
    )

In [243]:
plants_app[
    recommendation_boolean_columns
].dtypes

supports_full_sun                bool
supports_part_shade              bool
supports_shade                   bool
supports_low_water               bool
supports_medium_water            bool
supports_high_water              bool
bloom_jan                        bool
bloom_feb                        bool
bloom_mar                        bool
bloom_apr                        bool
bloom_may                        bool
bloom_jun                        bool
bloom_jul                        bool
bloom_aug                        bool
bloom_sep                        bool
bloom_oct                        bool
bloom_nov                        bool
bloom_dec                        bool
supports_bee_observed            bool
supports_butterfly_observed      bool
supports_moth_observed           bool
supports_fly_observed            bool
supports_beetle_observed         bool
supports_wasp_observed           bool
supports_hummingbird_observed    bool
has_ladybird_data                bool
has_pollinat

In [244]:
plants_app[
    recommendation_boolean_columns
].isna().sum()

supports_full_sun                0
supports_part_shade              0
supports_shade                   0
supports_low_water               0
supports_medium_water            0
supports_high_water              0
bloom_jan                        0
bloom_feb                        0
bloom_mar                        0
bloom_apr                        0
bloom_may                        0
bloom_jun                        0
bloom_jul                        0
bloom_aug                        0
bloom_sep                        0
bloom_oct                        0
bloom_nov                        0
bloom_dec                        0
supports_bee_observed            0
supports_butterfly_observed      0
supports_moth_observed           0
supports_fly_observed            0
supports_beetle_observed         0
supports_wasp_observed           0
supports_hummingbird_observed    0
has_ladybird_data                0
has_pollinator_data              0
dtype: int64

In [245]:
test4 = plants_app[
    plants_app["bloom_jul"]
]

print("Plants blooming in July:", len(test4))

test4[
    [
        "common_name",
        "scientific_name",
        "bloom_color",
    ]
].head(20)

Plants blooming in July: 421


,common_name,scientific_name,bloom_color
14,"Common Yarrow, Western Yarrow, Yarrow, Milfoil",Achillea millefolium,"White , Pink"
18,Black Baneberry,Actaea racemosa,"White , Green"
20,"Northern Maidenhair Fern, Maidenhair Fern",Adiantum pedatum,Not Applicable
24,Bottlebrush Buckeye,Aesculus parviflora,White
28,Slenderleaf False Foxglove,Agalinis tenuifolia,"White , Red , Pink , Yellow , Purple"
29,"Mosquito Plant, Mosquito-plant, Texas Hummingbird Mint",Agastache cana,Pink
30,"Blue Giant Hyssop, Blue Giant-hyssop, Fragrant Giant Hyssop, Lavender Hyssop, Anise Hyssop",Agastache foeniculum,"Blue , Purple"
31,Yellow Giant Hyssop,Agastache nepetoides,"Yellow , Green"
33,"American Century Plant, Century Plant",Agave americana,Yellow
35,"Parry's Agave, Century Plant, Parry Agave",Agave parryi,"Yellow , Green"


In [246]:
def is_native_to_state(value, state):
    if pd.isna(value):
        return False

    states = {
        item.strip()
        for item in str(value).split("|")
    }

    return state in states

In [247]:
plants_app.to_csv(
    PROCESSED_DIR / "plants_app.csv",
    index=False,
)

print(
    "Corrected application dataset saved:",
    PROCESSED_DIR / "plants_app.csv",
)

Corrected application dataset saved: C:\Users\hp\OneDrive\Desktop\biodiversity-planner\data\processed\plants_app.csv


In [248]:
print(
    "Image URL column exists:",
    "image_url" in plants_app.columns,
)

print(
    "Image source column exists:",
    "image_source" in plants_app.columns,
)

Image URL column exists: True
Image source column exists: True
